In [45]:
### load modules 
import numpy as np
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import re 
import pickle
import pingouin as pg
from itertools import groupby
from operator import itemgetter
from patsy import dmatrices
import statsmodels.formula.api as smf
import statsmodels.discrete.discrete_model as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

### load my scripts 
import data_combiner as dc
import plotting_funcs as pf
import sim as sim
#%pip install openpyxl
import openpyxl

plt.rcParams["figure.figsize"] = (20,10)
sns.set_context('talk',font_scale=2)
plt.rcParams["lines.linewidth"] = 3

plt.rcParams['font.size'] = 15
plt.rcParams['font.family'] = 'Verdana'

In [46]:
curr_round = 'round_1'
# all_filenames = glob.glob("/Users/rohinpalsule/Documents/GitHub/foraging_fMRI_pilot/experiment/experiment1k/run_exp/data/batch*/*.csv")
all_filenames = glob.glob("/Users/zyfxl/OneDrive - University of Southern California/Documents/GitHub/foraging_kids_4blocks_Yifan/data_analysis/data/foraging_pilot_data/*.csv")
# The filename order defines dc.clean_data_group sub_num (0-based).
all_filenames = sorted(all_filenames)
assert all_filenames, "No pilot CSV files found; check all_filenames path"


### verify essential functions working

In [47]:
import importlib
import sim
import data_combiner as dc

importlib.invalidate_caches()
importlib.reload(sim)
importlib.reload(dc)

# Verify both functions work
print("data_combiner:", len(dc.load_data()))
print("sim:", len(sim.load_data()))

data_combiner: 3
sim: 3


In [48]:
from pathlib import Path
import re

all_data = dc.clean_data_group(all_filenames).reset_index(drop=True)
all_decisions = dc.decision_trials_group(all_filenames)
num_subs = all_data["sub_num"].nunique()

all_data["prt_rel_mvt"] = all_data["prt"] - all_data["opt_prt"]
all_data["prt_rel_om"] = all_data["prt"] - all_data["opt_prt_om"]

# dc.clean_data_group enumerates all_filenames, assigning sub_num = 0, 1, ...
# Take the final five alphanumeric characters of the filename stem (e.g. 3580B).
sub_lookup = pd.DataFrame({
    "sub_num": range(len(all_filenames)),
    "subject_id": [Path(f).stem[-5:].upper() for f in all_filenames],
    "source_file": [Path(f).name for f in all_filenames]
})
assert sub_lookup["subject_id"].str.fullmatch(r"[A-Z0-9]{5}").all(), "Bad subject ID in filename"
assert sub_lookup["subject_id"].is_unique, "Duplicate subject IDs in filenames"

AGE_FILE = Path(
    r"C:\Users\zyfxl\OneDrive - University of Southern California"
    r"\Documents\GitHub\foraging_kids_4blocks_Yifan"
    r"\data_analysis\data_analysis\pilot_ages.xlsx"
)
if not AGE_FILE.exists():
    raise FileNotFoundError(f"Age sheet not found: {AGE_FILE}. Check your local file location.")
ages_raw = pd.read_excel(AGE_FILE, dtype=str)
print("Age spreadsheet columns:", ages_raw.columns.tolist())

# Accept common ID and age header variations, without relying on column order.
def _col(name):
    return re.sub(r"[^a-z0-9]", "", str(name).strip().lower())

columns = {_col(c): c for c in ages_raw.columns}
id_keys = ["subjectid", "participantid", "participant", "subject", "subid", "id"]
age_keys = ["age", "ageyears", "ageinyears", "participantage"]
id_col = next((columns[k] for k in id_keys if k in columns), None)
age_col = next((columns[k] for k in age_keys if k in columns), None)
if id_col is None or age_col is None:
    raise ValueError(f"Cannot infer ID/age columns from {ages_raw.columns.tolist()}. Set id_col and age_col explicitly.")

ages = ages_raw[[id_col, age_col]].copy()
ages.columns = ["subject_id", "age"]
ages["subject_id"] = (ages["subject_id"].astype("string")
                      .str.strip().str.upper().str.replace(r"\.0$", "", regex=True))
# Also accept a filename-style ID in the Excel file (e.g. final_S_3580B).
ages["subject_id"] = ages["subject_id"].str.extract(r"([A-Z0-9]{5})$", expand=False)
ages["age"] = pd.to_numeric(ages["age"], errors="coerce")
ages = ages.dropna(subset=["subject_id"])
if ages["subject_id"].duplicated().any():
    raise ValueError("Duplicate IDs in age spreadsheet: " +
                     str(ages.loc[ages.subject_id.duplicated(False), "subject_id"].tolist()))

sub_lookup = sub_lookup.merge(ages, on="subject_id", how="left", validate="one_to_one")
missing = sub_lookup.loc[sub_lookup["age"].isna(), ["sub_num", "subject_id", "source_file"]]
if not missing.empty:
    display(missing)
    raise ValueError("Missing or nonnumeric age for the above pilot subjects; check ID matching")

all_data = all_data.merge(sub_lookup[["sub_num", "subject_id", "age"]],
                          on="sub_num", how="left", validate="many_to_one")
all_decisions = all_decisions.merge(sub_lookup[["sub_num", "subject_id", "age"]],
                                    on="sub_num", how="left", validate="many_to_one")

# Utility for later groupby summaries (groupby(numeric_only=True) loses text IDs).
def add_subject_metadata(frame):
    return (frame.drop(columns=["subject_id", "age"], errors="ignore")
            .merge(sub_lookup[["sub_num", "subject_id", "age"]],
                   on="sub_num", how="left", validate="many_to_one"))

print("Raw decision RT quantiles:",
      pd.to_numeric(all_decisions["rt"], errors="coerce").quantile([.1, .5, .9]).to_dict())
display(sub_lookup[["sub_num", "subject_id", "age"]])

0


c:\Users\zyfxl\OneDrive - University of Southern California\Documents\GitHub\foraging_kids_4blocks_Yifan\data_analysis\data_analysis\data_combiner.py:351: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  clean_data = pd.concat([clean_data, clean_data_indiv(f,i)])


1
2
3
4
5
6
7
8
9


c:\Users\zyfxl\OneDrive - University of Southern California\Documents\GitHub\foraging_kids_4blocks_Yifan\data_analysis\data_analysis\data_combiner.py:355: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  clean_data['condition'] = clean_data['condition'].fillna(1)


Age spreadsheet columns: ['participant_id', 'age']
Raw decision RT quantiles: {0.1: 111.25999999046326, 0.5: 466.0, 0.9: 1239.0}


,sub_num,subject_id,age
0,0,2422A,12.825000
1,1,2422B,10.069444
2,2,3086A,12.386111
3,3,3276B,8.927778
4,4,3276C,8.936111
5,5,3562A,12.605556
6,6,3562B,9.066667
7,7,3567A,11.925000
8,8,3568A,12.530556
9,9,3580B,8.950000


## Process Age Data

In [ ]:
all_data

,sub_num,condition,age,gender,n_prac,block,planet,true_planet,prt,initial_reward,...,gal_2_trans_0,gal_2_trans_1,gal_2_trans_2,gal_0_encount,gal_1_encount,gal_2_encount,opt_prt,opt_prt_om,prt_rel_mvt,prt_rel_om
0,0,1,100,m,0,1,0,0,3,103,...,NaN,NaN,NaN,1,0,0,2,2,1,1
1,0,1,100,m,0,1,1,1,4,98,...,NaN,NaN,NaN,2,0,0,2,2,2,2
2,0,1,100,m,0,1,2,2,5,103,...,NaN,NaN,NaN,2,1,0,2,3,3,2
3,0,1,100,m,0,1,3,3,5,107,...,NaN,NaN,NaN,2,2,0,2,3,3,2
4,0,1,100,m,0,1,4,4,5,100,...,NaN,NaN,NaN,2,3,0,2,3,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350,9,1,100,m,0,3,16,43,2,98,...,0.166667,0.333333,0.5,5,6,6,4,4,-2,-2
351,9,1,100,m,0,4,17,60,21,98,...,0.166667,0.333333,0.5,6,6,6,6,2,15,19
352,9,1,100,m,0,4,18,61,10,100,...,0.166667,0.333333,0.5,7,6,6,3,2,7,8
353,9,1,100,m,0,4,19,62,24,94,...,0.166667,0.333333,0.5,8,6,6,8,2,16,22


In [ ]:
all_data["galaxy"] = all_data["galaxy"].astype("int32")
avgd_planet = add_subject_metadata(
    all_data.groupby(["sub_num", "true_planet"], as_index=False)[["galaxy", "prt", "prt_rel_om"]].mean()
)
# avgd_planet is a per-subject/per-planet summary; age and ID retained.


NameError: name 'add_subject_metadata' is not defined

In [ ]:
avgd_planet.head()

In [ ]:
avgd_data = add_subject_metadata(
    all_data.groupby("sub_num", as_index=False).mean(numeric_only=True)
)
mean = avgd_data["prt"].mean()
sd = avgd_data["prt"].std()
print(f"PRT ± 2 SD: {mean - 2*sd:.2f} to {mean + 2*sd:.2f}")

### Pilot exclusions (subject-level)

The historical paper's exclusion description lists comprehension failures **more than twice**, mean decision RT below 200 ms, and several extreme strategy criteria. In this pilot notebook, `catch_perf` is **the number of correct catch trials**, counted as `key_press == 90` by `data_combiner.catch_trial_perf`. Thus `catch_perf != 2` means the participant did **not get exactly 2 catch trials correct**; it is not the same measure as failing a comprehension check more than twice. Catch exclusion remains disabled below, as in the supplied notebook. Practice exclusion is also disabled. **New**: exclude participants aged 8 or 9 (8 ≤ age < 10). The existing active PRT cutoff of mean + 1 SD is retained; this differs from the markdown's ±2 SD historical criterion.

All downstream analyses operate on the filtered `all_data`, `all_decisions`, and `avgd_data`. Pre-exclusion copies are used only for exclusion diagnostics.

In [ ]:
# Compute criteria on the FULL sample, before removing anyone.
all_data_pre_exclusion = all_data.copy()
all_decisions_pre_exclusion = all_decisions.copy()
avgd_pre_exclusion = avgd_data.copy()

drop_catch = set(all_data.loc[all_data["catch_perf"] != 2, "sub_num"])
drop_low_prt = set(avgd_data.loc[avgd_data["prt"] < 1.5, "sub_num"])
mean, sd = avgd_data["prt"].mean(), avgd_data["prt"].std()
high_cutoff = mean + 1 * sd   # Preserve active notebook rule (not 2 SD)
drop_high_prt = set(avgd_data.loc[avgd_data["prt"] > high_cutoff, "sub_num"])
drop_practice = set(avgd_data.loc[avgd_data["n_prac"] > 2, "sub_num"])

rt_data = all_decisions[["sub_num", "rt"]].copy()
rt_data["rt"] = pd.to_numeric(rt_data["rt"], errors="coerce")
rt_data = rt_data.loc[np.isfinite(rt_data["rt"]) & (rt_data["rt"] > 0)].copy()
avg_rt_data = rt_data.groupby("sub_num", as_index=False)["rt"].mean()
drop_rt = set(avg_rt_data.loc[avg_rt_data["rt"] < 200, "sub_num"])

drop_age = set(sub_lookup.loc[sub_lookup["age"].between(8, 10, inclusive="left"), "sub_num"])

# Preserve which pre-existing exclusions were active in the uploaded notebook.
subs_to_drop = drop_low_prt | drop_high_prt | drop_rt | drop_age
# Optional: subs_to_drop |= drop_catch | drop_practice

for name, ids in {
    "Catch failure (informational; disabled)": drop_catch,
    "Low PRT": drop_low_prt,
    "High PRT": drop_high_prt,
    "Fast mean RT": drop_rt,
    "Practice (informational; disabled)": drop_practice,
    "Age 8–9": drop_age,
}.items():
    print(f"{name}: {len(ids)} {sorted(ids)}")
print(f"Unique excluded: {len(subs_to_drop)} {sorted(subs_to_drop)}")

all_data = all_data.loc[~all_data["sub_num"].isin(subs_to_drop)].copy()
all_decisions = all_decisions.loc[~all_decisions["sub_num"].isin(subs_to_drop)].copy()
avgd_data = avgd_data.loc[~avgd_data["sub_num"].isin(subs_to_drop)].copy()
num_subs = all_data["sub_num"].nunique()
retained_sub_nums = set(all_data["sub_num"].unique())
print(f"Retained subjects: {num_subs}; ages: {sorted(all_data.age.unique())}")

# Create trial-by-trial reward/RT export FROM FILTERED all_data.
# As in the source notebook, reward_list index is matched to rt_list index;
# this assignment should be validated against original event sequences.
rows = []
for p in all_data.itertuples(index=False):
    rewards = p.reward_list
    rts = p.rt_list
    for t, reward in enumerate(rewards):
        rows.append({"sub_num": p.sub_num, "subject_id": p.subject_id,
                     "age": p.age, "block": p.block,
                     "true_planet": p.true_planet, "planet": p.planet,
                     "galaxy": p.galaxy, "stay_num": t, "reward": reward,
                     "rt": rts[t] if t < len(rts) else np.nan})
temp = pd.DataFrame(rows)
# Index of planet within each subject/block, based on original planet order.
if not temp.empty:
    planet_order = (all_data.sort_values(["sub_num", "block", "planet"])
        .assign(planet_in_block=lambda d: d.groupby(["sub_num", "block"]).cumcount()))
    temp = temp.merge(
        planet_order[["sub_num", "block", "planet", "planet_in_block"]],
        on=["sub_num", "block", "planet"], how="left", validate="many_to_one"
    )
    temp = temp[["sub_num", "subject_id", "age", "block", "true_planet",
                 "planet", "galaxy", "stay_num", "reward", "rt", "planet_in_block"]]

# Filtered exports for later scripts and model-fitting inputs.
all_data.to_pickle("new_pilot_data_child.pkl")
all_data.to_csv("new_pilot_planet_child.csv", index=False)
all_decisions.to_csv("new_pilot_decisions_child.csv", index=False)
temp.to_csv("new_pilot_data_child.csv", index=False)


In [ ]:
# Check minimum PRT for each subject
prt_summary = all_data_pre_exclusion.groupby("sub_num")["prt"].agg(
    min_prt="min", 
    mean_prt="mean",
    n_planets="size",
    n_prt_1=lambda x: (x == 1).sum()
)

display(prt_summary)

print("Overall minimum PRT:", all_data_pre_exclusion["prt"].min())
print("Subjects with PRT = 1:",
      prt_summary.index[prt_summary["n_prt_1"] > 0].tolist())

In [ ]:
all_data

### Visualize dropping cut off

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Subject order sorted by mean
prt_order = avgd_pre_exclusion.sort_values("prt")["sub_num"].tolist()
rt_order = avg_rt_data.sort_values("rt")["sub_num"].tolist()

fig, axes = plt.subplots(
    1, 2, figsize=(18, max(12, len(prt_order) * 0.22)),
    constrained_layout=True
)

# PRT distribution for each subject
sns.boxplot(
    data=all_data_pre_exclusion, y="sub_num", x="prt",
    order=prt_order, orient="h",
    color="lightgray", fliersize=1, linewidth=0.6,
    ax=axes[0]
)

# Overlay subject means
prt_means = avgd_pre_exclusion.set_index("sub_num")["prt"]
for i, sub in enumerate(prt_order):
    axes[0].scatter(
        prt_means[sub], i,
        color="red" if sub in (drop_low_prt | drop_high_prt) else "royalblue",
        s=18, zorder=5
    )

axes[0].axvline(1.5, color="red", ls="--", lw=2, label="Low cutoff: 1.5")
axes[0].axvline(high_cutoff, color="darkorange", ls="--", lw=2,
                label=f"High cutoff: {high_cutoff:.2f}")
axes[0].set(title="Individual PRT distributions",
            xlabel="Planet residence time (PRT)", ylabel="Subject")
axes[0].legend()

# RT distribution for each subject
sns.boxplot(
    data=rt_data, y="sub_num", x="rt",
    order=rt_order, orient="h",
    color="lightgray", fliersize=1, linewidth=0.6,
    ax=axes[1]
)

rt_means = avg_rt_data.set_index("sub_num")["rt"]
for i, sub in enumerate(rt_order):
    axes[1].scatter(
        rt_means[sub], i,
        color="red" if sub in drop_rt else "royalblue",
        s=18, zorder=5
    )

axes[1].axvline(200, color="red", ls="--", lw=2,
                label="Low cutoff: 200 ms")
axes[1].set(title="Individual RT distributions",
            xlabel="Reaction time (ms)", ylabel="Subject")
axes[1].legend()

for ax in axes:
    ax.tick_params(axis="y", labelsize=6)
    sns.despine(ax=ax)

plt.show()

## Cross studies comparison

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
palette = {"Harhen 25": "#4477AA", "Current pilot": "#CC6677"}

# ============================================================
# 1. LOAD DATA
# ============================================================

path = Path(
    r"C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab"
    r"\Foraging Task Overview\Harhen-Budiono-Hartley-Bornstein-2025-Foraging-main"
    r"\data_analysis\transformed_data"
)

old = pd.read_csv(path / "linear_df_20241221.csv")
old_rt = pd.read_csv(path / "stay_num_RT_20241221.csv")

old = old.loc[old["age_group"].eq("child")].copy()
old_rt = old_rt.loc[old_rt["age_group"].eq("child")].copy()
pilot = all_data.copy()

# ============================================================
# 2. HARMONIZE PLANET-LEVEL DATA
# ============================================================

old["study"] = "Harhen 25"
pilot["study"] = "Current pilot"

old["prt_mvt"] = old["prt_rel_om"]
pilot["prt_mvt"] = pilot["prt"] - pilot["opt_prt_om"]

old["switch"] = old["switch"].astype(str).str.lower()

pilot = pilot.sort_values(["sub_num", "block", "planet"]).copy()
pilot["prev_galaxy"] = pilot.groupby(
    ["sub_num", "block"]
)["galaxy"].shift()

pilot["switch"] = np.where(
    pilot["prev_galaxy"].notna() &
    pilot["galaxy"].ne(pilot["prev_galaxy"]),
    "switch", "no_switch"
)

planets = pd.concat([
    old[["study", "sub_num", "prt", "prt_mvt"]],
    pilot[["study", "sub_num", "prt", "prt_mvt"]]
], ignore_index=True)

# Subject-level means to avoid pseudoreplication
subject_prt = (
    planets.groupby(["study", "sub_num"], as_index=False)
    [["prt", "prt_mvt"]].mean()
)

# ============================================================
# 3. FIRST DECISION RT AND CHOICE
# ============================================================

# Historical file contains the first decision per recorded planet
old_first = old_rt.loc[old_rt["stay_num"] == 1].copy()
old_first["study"] = "Harhen 25"
old_first["stay"] = pd.to_numeric(old_first["stay?"], errors="coerce")
old_first["rt"] = pd.to_numeric(old_first["rt"], errors="coerce")
old_first["switch"] = old_first["switch"].astype(str).str.lower()

# Pilot first decision from each planet's RT list
pilot_first = pilot.copy()

pilot_first["rt"] = pilot_first["rt_list"].apply(
    lambda x: x[0] if isinstance(x, (list, tuple, np.ndarray))
    and len(x) > 0 else np.nan
)

# For pilot, PRT=1 means leaving after the first opportunity
pilot_first["stay"] = (pilot_first["prt"] > 1).astype(int)

first = pd.concat([
    old_first[["study", "sub_num", "switch", "rt", "stay"]],
    pilot_first[["study", "sub_num", "switch", "rt", "stay"]]
], ignore_index=True)

first["rt"] = pd.to_numeric(first["rt"], errors="coerce")
first = first.loc[
    np.isfinite(first["rt"]) & (first["rt"] > 0)
].copy()

first = first.loc[
    first["switch"].isin(["switch", "no_switch"])
].copy()

# Average within participant and condition
first_subject = (
    first.groupby(
        ["study", "sub_num", "switch"], as_index=False
    )
    .agg(
        mean_rt=("rt", "mean"),
        stay_prob=("stay", "mean"),
        n_decisions=("stay", "size")
    )
)

print("Historical children:", old.sub_num.nunique())
print("Pilot subjects:", pilot.sub_num.nunique())
print("Historical first decisions:", len(old_first))
print("Pilot first decisions:", len(pilot_first))

# ============================================================
# 4. RECONSTRUCT PILOT TRIAL-LEVEL BEHAVIOR
# ============================================================

# Use only planets with explicit n rewards + 1 decision RTs.
# Otherwise reward-to-decision alignment is ambiguous.
rows = []
mismatch = []

for _, p in pilot.iterrows():

    rewards = p["reward_list"]
    rts = p["rt_list"]

    if not isinstance(rewards, (list, tuple, np.ndarray)):
        continue
    if not isinstance(rts, (list, tuple, np.ndarray)):
        continue

    rewards = list(rewards)
    rts = list(rts)

    if len(rts) != len(rewards) + 1:
        mismatch.append({
            "sub_num": p["sub_num"],
            "planet": p["planet"],
            "n_rewards": len(rewards),
            "n_rt": len(rts)
        })
        continue

    # One arrival reward, then harvested rewards
    decision_rewards = [p["initial_reward"]] + rewards
    decisions = [0] * len(rewards) + [1]  # 0=stay, 1=leave

    for t, (reward, rt, leave) in enumerate(
        zip(decision_rewards, rts, decisions)
    ):
        rows.append({
            "sub_num": p["sub_num"],
            "block": p["block"],
            "planet": p["planet"],
            "switch": p["switch"],
            "trial": t + 1,
            "reward": reward,
            "rt": rt,
            "leave": leave,
            "action": "leave" if leave else "stay"
        })

trial = pd.DataFrame(rows)
mismatch = pd.DataFrame(mismatch)

if not trial.empty:
    trial["reward"] = pd.to_numeric(trial["reward"], errors="coerce")
    trial["rt"] = pd.to_numeric(trial["rt"], errors="coerce")
    trial = trial.loc[
        np.isfinite(trial["reward"]) &
        np.isfinite(trial["rt"]) &
        (trial["rt"] > 0)
    ].copy()

print("\nTrial reconstruction:")
print("Matched decisions:", len(trial))
print("Unmatched planets:", len(mismatch))

if len(mismatch):
    display(mismatch.head(10))

# ============================================================
# 5. VISUALIZATIONS
# ============================================================

fig, axes = plt.subplots(3, 2, figsize=(14, 16))
order = ["Harhen 25", "Current pilot"]
switch_order = ["no_switch", "switch"]

# A. Raw PRT
sns.boxplot(
    data=subject_prt, x="study", y="prt",
    order=order, color="lightgray",
    showfliers=False, ax=axes[0, 0]
)
sns.stripplot(
    data=subject_prt, x="study", y="prt",
    order=order, hue="study", palette=palette,
    jitter=0.18, alpha=0.6, size=4,
    legend=False, ax=axes[0, 0]
)
axes[0, 0].set(
    title="A. Raw PRT",
    xlabel="", ylabel="Mean PRT per subject"
)

# B. PRT relative to MVT
sns.boxplot(
    data=subject_prt, x="study", y="prt_mvt",
    order=order, color="lightgray",
    showfliers=False, ax=axes[0, 1]
)
sns.stripplot(
    data=subject_prt, x="study", y="prt_mvt",
    order=order, hue="study", palette=palette,
    jitter=0.18, alpha=0.6, size=4,
    legend=False, ax=axes[0, 1]
)
axes[0, 1].axhline(0, color="black", ls="--", alpha=0.5)
axes[0, 1].set(
    title="B. PRT relative to MVT",
    xlabel="", ylabel="Mean PRT - MVT"
)

# C. First-decision RT vs switch
sns.pointplot(
    data=first_subject, x="switch", y="mean_rt",
    hue="study", order=switch_order,
    hue_order=order, palette=palette,
    errorbar=("ci", 95), dodge=0.2,
    markers="o", capsize=0.12,
    ax=axes[1, 0]
)
axes[1, 0].set(
    title="C. First-decision RT by galaxy switch",
    xlabel="Galaxy transition", ylabel="Mean RT (ms)"
)

# D. Probability of staying on first decision
sns.pointplot(
    data=first_subject, x="switch", y="stay_prob",
    hue="study", order=switch_order,
    hue_order=order, palette=palette,
    errorbar=("ci", 95), dodge=0.2,
    markers="o", capsize=0.12,
    ax=axes[1, 1]
)
axes[1, 1].set(
    title="D. First-decision stay probability",
    xlabel="Galaxy transition", ylabel="P(stay)"
)
axes[1, 1].set_ylim(0, 1)

# E and F. Pilot reward-dependent trial behavior
if not trial.empty:
    # Bin rewards for robust visualization
    trial["reward_bin"] = pd.cut(
        trial["reward"], bins=12, duplicates="drop"
    )
    trial["reward_mid"] = trial["reward_bin"].apply(
        lambda x: x.mid
    ).astype(float)

    # Aggregate within subject to avoid treating trials as subjects
    by_subject = (
        trial.groupby(
            ["sub_num", "reward_mid"], as_index=False
        )
        .agg(p_leave=("leave", "mean"))
    )

    sns.lineplot(
        data=by_subject, x="reward_mid", y="p_leave",
        marker="o", errorbar=("ci", 95),
        color=palette["Current pilot"],
        ax=axes[2, 0]
    )

    axes[2, 0].set(
        title="E. Leaving probability vs reward (pilot)",
        xlabel="Reward received", ylabel="P(leave)"
    )
    axes[2, 0].set_ylim(0, 1)

    # Mean RT by reward and choice, first within subject
    rt_by_subject = (
        trial.groupby(
            ["sub_num", "reward_mid", "action"], as_index=False
        )["rt"].mean()
    )

    sns.lineplot(
        data=rt_by_subject,
        x="reward_mid", y="rt", hue="action",
        hue_order=["stay", "leave"],
        marker="o", errorbar=("ci", 95),
        ax=axes[2, 1]
    )

    axes[2, 1].set(
        title="F. RT vs reward and choice (pilot)",
        xlabel="Reward received", ylabel="Mean RT (ms)"
    )
else:
    for ax in axes[2]:
        ax.text(
            0.5, 0.5,
            "Trial alignment unavailable",
            ha="center", va="center",
            transform=ax.transAxes
        )
        ax.set_axis_off()

plt.tight_layout()
plt.show()

# ============================================================
# 6. SUMMARY TABLES AND EXPORT
# ============================================================

print("\nPRT comparison:")
display(
    subject_prt.groupby("study")[["prt", "prt_mvt"]]
    .agg(["mean", "std", "count"]).round(2)
)

print("\nFirst-decision comparison:")
display(
    first_subject.groupby(["study", "switch"])[
        ["mean_rt", "stay_prob"]
    ].agg(["mean", "std", "count"]).round(3)
)

subject_prt.to_csv("comparison_subject_prt.csv", index=False)
first_subject.to_csv("comparison_first_decisions.csv", index=False)

if not trial.empty:
    trial.to_csv("pilot_trial_reward_rt.csv", index=False)

In [ ]:
len(np.unique(all_data.sub_num))

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
sns.set_context('talk',font_scale=2)
avgd_data = all_data.groupby(by=['sub_num','galaxy']).mean(numeric_only=True).reset_index()
avgd_data = add_subject_metadata(avgd_data)
sns.swarmplot(x='galaxy',y='prt',palette='Set1',size=10,alpha=0.5,edgecolor='k',data=avgd_data)

g=sns.boxplot(x='galaxy',y='prt',color='grey',data=avgd_data)
g.set_yticks(np.arange(0,25,5))
g.set_ylabel("PRT")
g.set_xlabel("planet type")
g.set_xticklabels(["poor","neutral","rich"])


#plt.savefig('plots/planet_type_prt.png') #optional save

In [ ]:
avgd_data = all_data.groupby(by=['sub_num']).mean(numeric_only=True).reset_index()
avgd_data = add_subject_metadata(avgd_data)
plt.rcParams["figure.figsize"] = (20,15)

sns.histplot(x='prt_rel_om',color='grey',data=avgd_data)
sns.despine(top=True,right=True)
plt.xlabel('PRT relative to MVT')
plt.plot([0,0],[0,30],'k--',linewidth=10)
plt.tight_layout()

plt.savefig('plots/avg_over_gal_indiv_sub.png')

In [ ]:
avgd_data = all_data.groupby(by=['sub_num']).mean(numeric_only=True).reset_index()
avgd_data = add_subject_metadata(avgd_data)
stats.ttest_1samp(avgd_data.prt_rel_om,popmean=0)

In [ ]:
len(np.unique(avgd_data.sub_num))

In [ ]:
def label_period(row):
    if row['block'] ==1:
        return 'early'
    elif row['block'] ==2:
        return 'early'
    elif row['block'] ==3:
        return 'late'
    elif row['block'] ==4:
        return 'late'

In [ ]:
all_data['period'] = all_data.apply (lambda row: label_period(row), axis=1)
plt.rcParams["figure.figsize"] = (10,10)
sns.set_context('talk',font_scale=2.5)


avgd_data = all_data.groupby(by=['sub_num','period']).mean(numeric_only=True).reset_index()
avgd_data = add_subject_metadata(avgd_data)
#avgd_data=avgd_data.query("sub_num in @subs")
sns.swarmplot(x='period',y='prt_rel_om',color='white',edgecolor="black",linewidth=2,alpha=0.2,size=10,data=avgd_data.query(" period !='med'"))
sns.boxplot(x='period',y='prt_rel_om',color='grey',showmeans=True,
            meanprops={"marker":"P",
                       "markerfacecolor":"white", 
                       "markeredgecolor":"black",
                      "markersize":"15"},data=avgd_data.query(" period !='med'"))
plt.ylabel("PRT relative to MVT")
plt.xlabel("block")
plt.plot([-0.5,2.5],[0,0],'k--')

plt.savefig('plots/block_period.png',bbox_inches = "tight",dpi=300)

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
sns.set_context('talk',font_scale=2.5)

avgd_data = all_data.groupby(by=['sub_num','galaxy']).mean(numeric_only=True).reset_index()
avgd_data = add_subject_metadata(avgd_data)
sns.swarmplot(x='galaxy',y='prt_rel_om',palette='Set1',size=10,alpha=0.5,edgecolor='k',data=avgd_data)
g=sns.boxplot(x='galaxy',y='prt_rel_om',color='grey',data=avgd_data,showmeans=True,
            meanprops={"marker":"P",
                       "markerfacecolor":"white", 
                       "markeredgecolor":"black",
                      "markersize":"15"})
plt.ylabel("PRT relative to MVT")
plt.xlabel("planet type")
g.set_xticklabels(['poor','neutral','rich'])
plt.yticks([-8,-4,0,4,8])
plt.plot([-0.5,2.5],[0,0],'k--')
plt.savefig('plots/overharvesting_galaxy.png',bbox_inches = "tight",dpi=300)

In [ ]:
avgd_data

In [ ]:
avgd_data['sub_num'].unique()

In [ ]:
avgd_data = all_data.groupby(by=['sub_num','galaxy']).mean(numeric_only=True).reset_index()
avgd_data = add_subject_metadata(avgd_data)
sns.swarmplot(x='galaxy',y='prt_rel_om',palette='Set1',size=10,alpha=0.3,edgecolor='k',data=avgd_data)
g=sns.lineplot(x='galaxy',y='prt_rel_om',color='mediumpurple',data=avgd_data,err_style='bars',errorbar='se',linewidth=5,err_kws={'elinewidth': 5})
plt.ylabel("PRT relative to MVT")
plt.xlabel("planet type")
g.set_xticklabels(['poor','neutral','rich'])
plt.ylim(-2,2)
plt.plot([-0.5,2.5],[0,0],'k--')
# plt.savefig('plots/overharvesting_galaxy.png',bbox_inches = "tight",dpi=300)

## CHILD FORAGING: BEHAVIOR × MODEL FITTING ANALYSIS

In [ ]:
# ============================================================
# CHILD FORAGING: BEHAVIOR × MODEL FITTING ANALYSIS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from IPython.display import display

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------


FIT_DIR = Path(
    r"C:\Users\zyfxl\OneDrive - University of Southern California"
    r"\Documents\GitHub\foraging_kids_4blocks_Yifan"
    r"\modeling\alpha_model_results"
)

FIG_DIR = FIT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Check available model outputs
for model in ["alpha_0", "alpha_1"]:
    path = FIT_DIR / model / "summary.csv"
    print(f"{model}: {'Found' if path.exists() else 'MISSING'}")
    print(f"  {path}")

# Load model fits
fit0 = pd.read_csv(FIT_DIR / "alpha_0" / "summary.csv")
fit1 = pd.read_csv(FIT_DIR / "alpha_1" / "summary.csv")
# Keep model fits for the same post-exclusion cohort only.
fit0 = fit0.loc[fit0["sub_num"].isin(retained_sub_nums)].copy()
fit1 = fit1.loc[fit1["sub_num"].isin(retained_sub_nums)].copy()
behavior = all_data.copy()

# ------------------------------------------------------------
# 2. Subject-level PRT and overharvesting
# ------------------------------------------------------------

# 0 = Poor, 1 = Neutral, 2 = Rich
galaxy_names = {0: "Poor", 1: "Neutral", 2: "Rich"}
galaxies = ["Poor", "Neutral", "Rich"]

behavior["galaxy"] = behavior["galaxy"].astype(int).map(galaxy_names)

# Average across planets within each subject and galaxy
bg = (
    behavior.groupby(["sub_num", "galaxy"])[["prt", "prt_rel_om"]]
    .mean()
    .reset_index()
)

wide = bg.pivot(
    index="sub_num",
    columns="galaxy",
    values=["prt", "prt_rel_om"]
)

wide.columns = [
    f"{'PRT' if metric == 'prt' else 'Overharvest'}_{galaxy}"
    for metric, galaxy in wide.columns
]

wide = wide.reset_index()
wide = add_subject_metadata(wide)

# Equal-weighted average across the three planet types
for prefix in ["PRT", "Overharvest"]:
    wide[f"{prefix}_Overall"] = wide[
        [f"{prefix}_{g}" for g in galaxies]
    ].mean(axis=1, skipna=False)

# Attach both model fits by subject ID
def prefix_fits(df, model):
    assert df["sub_num"].is_unique
    return df.rename(
        columns={c: f"{model}_{c}" for c in df.columns if c != "sub_num"}
    )

df = (
    wide
    .merge(prefix_fits(fit0, "alpha_0"), on="sub_num",
           how="inner", validate="one_to_one")
    .merge(prefix_fits(fit1, "alpha_1"), on="sub_num",
           how="inner", validate="one_to_one")
)

print(f"Subjects with both model fits: {len(df)}")
assert len(df) == len(wide), "Some subjects are missing model fits"

df.to_csv(FIT_DIR / "behavior_model_merged.csv", index=False)

# ------------------------------------------------------------
# 3. Behavioral visualization
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, metric, ylabel in zip(
    axes,
    ["prt", "prt_rel_om"],
    ["PRT", "PRT relative to MVT"]
):
    # Individual subject trajectories
    for _, sub in bg.groupby("sub_num"):
        y = sub.set_index("galaxy")[metric].reindex(galaxies)
        ax.plot(galaxies, y, "o-", color="gray", alpha=0.35, lw=1)

    # Group mean +/- SEM
    summary = bg.groupby("galaxy")[metric].agg(["mean", "sem"])
    summary = summary.reindex(galaxies)

    ax.errorbar(
        galaxies, summary["mean"], yerr=summary["sem"],
        fmt="o-", color="black", lw=2.5, capsize=5
    )

    if metric == "prt_rel_om":
        ax.axhline(0, ls="--", color="gray")

    ax.set(xlabel="Planet type", ylabel=ylabel)
    sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(FIG_DIR / "behavior_by_galaxy.png", dpi=300)
plt.show()

# ------------------------------------------------------------
# 4. Spearman: behavior vs fitted model parameters
# ------------------------------------------------------------

outcomes = [
    f"{prefix}_{g}"
    for prefix in ["PRT", "Overharvest"]
    for g in galaxies + ["Overall"]
]

# Only include genuinely free parameters
parameters = {
    "α0: γ_base": "alpha_0_gamma_base",
    "α0: epsilon": "alpha_0_epsilon",
    "α1: γ_base": "alpha_1_gamma_base",
    "α1: γ_coef": "alpha_1_gamma_coef",
    "α1: epsilon": "alpha_1_epsilon",
}

results = []

for label, param in parameters.items():
    for outcome in outcomes:
        sub = df[[param, outcome]].apply(
            pd.to_numeric, errors="coerce"
        ).dropna()

        if (len(sub) >= 4 and
            sub[param].nunique() > 1 and
            sub[outcome].nunique() > 1):
            rho, p = spearmanr(sub[param], sub[outcome])
        else:
            rho, p = np.nan, np.nan

        results.append({
            "Parameter": label,
            "Outcome": outcome,
            "N": len(sub),
            "rho": rho,
            "p": p
        })

corr = pd.DataFrame(results)
corr["p_fdr"] = np.nan

valid = corr["p"].notna()
if valid.any():
    corr.loc[valid, "p_fdr"] = multipletests(
        corr.loc[valid, "p"], method="fdr_bh"
    )[1]

corr.to_csv(FIT_DIR / "behavior_parameter_correlations.csv", index=False)

print("\nSpearman correlations (sorted by p-value):")
display(corr.sort_values("p").round(4).head(20))

# Correlation heatmap
matrix = corr.pivot(
    index="Outcome", columns="Parameter", values="rho"
).reindex(index=outcomes, columns=list(parameters))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    matrix, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title("Spearman: Behavior × Model Parameters")
fig.tight_layout()
fig.savefig(FIG_DIR / "behavior_parameter_heatmap.png", dpi=300)
plt.show()

# ------------------------------------------------------------
# 5. Selected parameter-behavior scatterplots
# ------------------------------------------------------------

pairs = [
    ("alpha_1_gamma_base", "PRT_Overall"),
    ("alpha_1_gamma_coef", "Overharvest_Overall"),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, (x, y) in zip(axes, pairs):
    sub = df[[x, y]].dropna()

    sns.regplot(
        data=sub, x=x, y=y, ax=ax,
        scatter_kws={"s": 65, "alpha": 0.8},
        line_kws={"color": "red"},
        ci=None
    )

    rho, p = spearmanr(sub[x], sub[y])

    ax.set_title(f"Spearman ρ = {rho:.3f}, p = {p:.3g}")
    ax.set_xlabel(x.replace("alpha_1_", ""))
    ax.set_ylabel(y.replace("_", " "))
    sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(FIG_DIR / "parameter_behavior_scatter.png", dpi=300)
plt.show()

# ------------------------------------------------------------
# 6. Model comparison: AIC and BIC
# ------------------------------------------------------------

comparison = df[[
    "sub_num", "subject_id", "age",
    "alpha_0_AIC", "alpha_1_AIC",
    "alpha_0_BIC", "alpha_1_BIC"
]].copy()

comparison["Delta_AIC"] = (
    comparison["alpha_1_AIC"] - comparison["alpha_0_AIC"]
)

comparison["Delta_BIC"] = (
    comparison["alpha_1_BIC"] - comparison["alpha_0_BIC"]
)

comparison = comparison.sort_values("sub_num")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, metric in zip(axes, ["Delta_AIC", "Delta_BIC"]):
    values = comparison[metric]
    colors = np.where(values < 0, "steelblue", "coral")

    ax.barh(
        comparison["sub_num"].astype(str),
        values, color=colors
    )
    ax.axvline(0, color="black", linestyle="--")
    ax.set_xlabel(f"{metric.replace('_', ' ')} (α1 − α0)")
    ax.set_ylabel("Subject")
    ax.set_title(f"{(values < 0).sum()}/{len(values)} favor α1")
    sns.despine(ax=ax)

fig.suptitle("Model Fit Comparison (negative favors α1)", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "model_comparison.png", dpi=300)
plt.show()

comparison.to_csv(FIT_DIR / "behavior_model_comparison.csv", index=False)

print("\nMODEL COMPARISON SUMMARY")
print(f"Mean ΔAIC: {comparison.Delta_AIC.mean():.3f}")
print(f"Mean ΔBIC: {comparison.Delta_BIC.mean():.3f}")
print(f"α1 preferred by AIC: {(comparison.Delta_AIC < 0).sum()}/{len(comparison)}")
print(f"α1 preferred by BIC: {(comparison.Delta_BIC < 0).sum()}/{len(comparison)}")

# ------------------------------------------------------------
# 7. Parameter distributions
# ------------------------------------------------------------

fits_long = pd.concat([
    fit0.assign(Model="alpha_0"),
    fit1.assign(Model="alpha_1")
], ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Parameters estimated in both models
for ax, param in zip(axes[:2], ["gamma_base", "epsilon"]):
    sns.boxplot(
        data=fits_long, x="Model", y=param,
        order=["alpha_0", "alpha_1"],
        showfliers=False, ax=ax
    )
    sns.stripplot(
        data=fits_long, x="Model", y=param,
        order=["alpha_0", "alpha_1"],
        color="black", size=6, alpha=0.7, ax=ax
    )
    ax.set_title(param)
    sns.despine(ax=ax)

# gamma_coef is only fitted in alpha_1
sns.stripplot(
    data=fit1.assign(Model="alpha_1"),
    x="Model", y="gamma_coef",
    color="black", size=7, jitter=0.15, ax=axes[2]
)
axes[2].axhline(0, ls="--", color="gray")
axes[2].set_title("gamma_coef (α1 only)")
sns.despine(ax=axes[2])

fig.suptitle("Fitted Parameter Distributions", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "parameter_distributions.png", dpi=300)
plt.show()

print("\nPARAMETER DESCRIPTIVES")
display(
    fits_long.groupby("Model")[["gamma_base", "gamma_coef", "epsilon"]]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

In [ ]:
avgd_data = all_data.groupby(by=['sub_num','period']).mean(numeric_only=True).reset_index()
avgd_data = add_subject_metadata(avgd_data)

x=avgd_data.query("period=='early'").reset_index().prt_rel_om
y=avgd_data.query("period=='late'").reset_index().prt_rel_om

stats.ttest_rel(x,y)

In [ ]:
avgd_data=avgd_data.query("sub_num not in @subs_to_drop & period !='med'")

x=avgd_data.query("period=='early'").reset_index().prt_rel_om
y=avgd_data.query("period=='late'").reset_index().prt_rel_om
z=x-y
stats.ttest_rel(x,y)

In [ ]:
avgd_data = all_data.groupby(by=['sub_num']).mean(numeric_only=True).reset_index()
avgd_data = add_subject_metadata(avgd_data)

stats.ttest_1samp(avgd_data.prt_rel_om,popmean=0)

In [ ]:
plt.rcParams["figure.figsize"] = (20,15)
avgd_data = all_data.groupby(by=['sub_num','galaxy']).mean(numeric_only=True).reset_index()
avgd_data = add_subject_metadata(avgd_data)

fig, ax = plt.subplots()
sns.barplot(x='galaxy',y='prt_rel_om',ci=68,color='grey',data=avgd_data,ax=ax)

g=sns.pointplot(x='galaxy',y='prt_rel_om',hue='sub_num',ci=68,palette='hls',data=all_data.query("sub_num!='nan'"),ax=ax)

g.set_xticklabels(['poor','neutral','rich'])
g.set_ylabel('PRT relative to optimal')
g.set_xlabel('planet type')
sns.despine(top=True,right=True)
plt.setp(g.collections, alpha=.5) #for the markers
plt.setp(g.lines, alpha=.5)       #for the lines
g.get_legend().remove()
plt.plot([-0.5,2.5],[0,0],'k--', linewidth=5)


plt.tight_layout()
#plt.savefig('plots/galaxy_sep_indiv_sub_2022.png')

In [ ]:
all_data.columns

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)

sns.set_context('talk',font_scale=2.5)
avgd_data = all_data.query("galaxy!=2").groupby(by=['sub_num','switch_point']).mean(numeric_only=True).reset_index()
avgd_data = add_subject_metadata(avgd_data)
sns.swarmplot(x='switch_point',y='prt_rel_om',color='white',edgecolor="black",linewidth=2,alpha=0.2,size=10,data=avgd_data)
g=sns.boxplot(x='switch_point',y='prt_rel_om',color='grey',data=avgd_data,showmeans=True,
            meanprops={"marker":"P",
                       "markerfacecolor":"white", 
                       "markeredgecolor":"black",
                      "markersize":"15"})
plt.ylabel("PRT relative to MVT")
plt.xlabel("")
g.set_xticklabels(['no switch','switch'],)
plt.plot([-0.5,2.5],[0,0],'k--')

plt.savefig('plots/overharveting uncertainty.png',bbox_inches = "tight",dpi=300)

In [ ]:
subs = np.unique(all_data.sub_num)
x=[]
for sub in subs:
    temp = all_data.query("sub_num==@sub & galaxy==2").prt_rel_om
    t,p = stats.ttest_1samp(temp,popmean=0)
    
    if p < 0.05:
        if t <0:
            x+=[-1]
        else:
            x+=[1]
    else:
        x+=[0]
        
x

In [ ]:
len(np.where(np.array(x)==1)[0])

In [ ]:
len(np.where(np.array(x)==0)[0])

In [ ]:
len(np.where(np.array(x)==-1)[0])

In [ ]:
avgd_data = all_data.groupby(by=['sub_num','galaxy']).mean().reset_index()
avgd_data = add_subject_metadata(avgd_data)

In [ ]:
print(stats.ttest_rel(avgd_data.query('galaxy==2').prt,avgd_data.query('galaxy==1').prt))

print(stats.ttest_rel(avgd_data.query('galaxy==1').prt,avgd_data.query('galaxy==0').prt))

In [ ]:
print(stats.ttest_1samp(avgd_data.query('galaxy==0').prt_rel_om,popmean=0))
print(stats.ttest_1samp(avgd_data.query('galaxy==1').prt_rel_om,popmean=0))
print(stats.ttest_1samp(avgd_data.query('galaxy==2').prt_rel_om,popmean=0))

In [ ]:
plt.rcParams["figure.figsize"] = (20,15)

g=sns.pointplot(x='galaxy',y='prt_rel_om',ci=68,color='grey',data=avgd_data.query("sub_num!='nan'"))
g.set_xticklabels(['poor','neutral','rich'])
g.set_ylabel('PRT rel Opt')

plt.tight_layout()
plt.savefig('plots/galaxy_sep_avg.png')

In [ ]:
plt.rcParams["figure.figsize"] = (20,10)

temp=all_data.groupby(by=['sub_num','galaxy','preced_gal','num_in_gal']).prt_rel_om.mean().reset_index()
sns.pointplot(x='num_in_gal',y='prt_rel_om',hue='preced_gal',hue_order=[0,2],palette=['r','g'],ci=68,aspect=10,data=temp.query('galaxy==1 &(preced_gal==0 | preced_gal==2) &num_in_gal <=3'))
plt.ylabel("PRT relative to MVT")
plt.xlabel("Planet in neutral galaxy")

plt.legend().remove()
savefig=1
#plt.tight_layout()
plt.savefig("plots/recency_effects_neutral_only_2021.png")

In [ ]:
plt.rcParams["figure.figsize"] = (20,10)

temp=all_data.groupby(by=['sub_num','galaxy','preced_gal','num_in_gal']).prt_rel_om.mean().reset_index()
sns.pointplot(x='num_in_gal',y='prt_rel_om',hue='preced_gal',hue_order=[1,2],palette=['b','g'],ci=68,aspect=10,data=temp.query('galaxy==0 &(preced_gal==1 | preced_gal==2)  &num_in_gal <=3'))
plt.ylabel("PRT relative to MVT")
plt.xlabel("Planet in poor galaxy")

plt.legend().remove()
savefig=1
#plt.tight_layout()
if savefig:
    plt.savefig("plots/recency_effects_poor_only_2021.png")

# Reaction Time check 

In [ ]:
def make_sub_rt_df(sub_num):
    print(sub_num)
    sub_df = pd.DataFrame(columns=['sub_num','condition','block','planet_num','stay_num','stay?','galaxy','prior_galaxy','num_in_gal'])
    curr_sub = all_data.query("sub_num=="+str(sub_num)).reset_index()
    n_planet = max(curr_sub.planet) + 1
    cond_num = curr_sub.condition[0]

    for p in range(n_planet):
        curr_planet = curr_sub.query("planet=="+str(p)).reset_index()
        curr_block = curr_planet.block[0]
        curr_galaxy = curr_planet.galaxy[0]
        curr_prior_galaxy = curr_planet.preced_gal[0]
        num_in_gal = curr_planet.num_in_gal[0]
        rt = curr_planet.rt_list[0]
        n_stay = len(rt)
        
        if n_stay == 0:
            rt = [np.nan]
            n_stay = 1
        tmp = pd.DataFrame({'sub_num':[sub_num]*n_stay, 'condition':[cond_num]*n_stay, 
                        'block':[curr_block]*n_stay,'planet_num':[p]*n_stay,'stay_num':range(n_stay),'stay?':[1]*(n_stay-1) + [0],
                        'galaxy':[curr_galaxy]*n_stay,'prior_galaxy':[curr_prior_galaxy]*n_stay,'num_in_gal':[num_in_gal]*n_stay,
                       'rt':rt})
        sub_df = pd.concat([sub_df,tmp])
    sub_df = sub_df.query("rt!=0")
    sub_df['zlog_rt'] = stats.zscore(np.log(sub_df['rt']),nan_policy='omit')
    return sub_df

def make_rt_df(subs):
    rt_df = pd.DataFrame(columns=['sub_num','condition','block','planet_num','stay_num','stay?','galaxy','prior_galaxy','num_in_gal'])

    for sub in subs:
        sub_df = make_sub_rt_df(sub)
        rt_df = pd.concat([rt_df,sub_df])
    return rt_df

In [ ]:
subs = list(set(all_data.sub_num.tolist()))
rt_df = make_rt_df(subs)

In [ ]:
len(np.unique(rt_df.sub_num))

In [ ]:
def label_switch(row):
    if row['num_in_gal'] ==1:
        return 'switch'
    else:
        return 'no_switch'
    
def label_group(row):
    if row['alpha'] <0.8:
        return 'below'
    else:
        return 'above'
    

In [ ]:
rt_df['switch'] = rt_df.apply (lambda row: label_switch(row), axis=1)
all_data['switch'] =  all_data.apply (lambda row: label_switch(row), axis=1)

In [ ]:
x=all_data.groupby(by=['sub_num','switch']).mean(numeric_only=True).prt_rel_om.reset_index()

y=x.query("switch=='no_switch'").prt_rel_om.reset_index(drop=True)
z=x.query("switch=='switch'").prt_rel_om.reset_index(drop=True)
diff=z-y
stats.ttest_1samp(diff,popmean=0)

In [ ]:
avg_switch=rt_df.query("stay_num==1").groupby(by=['sub_num','switch']).mean(numeric_only=True).reset_index()
#subs_to_drop += [192,177,163,158,110,99,94,29,78,2,39,77,106,143,169,175,48,64,8,22,34,164]

#avg_switch=avg_switch.query("sub_num in @subs")

In [ ]:
pal=sns.color_palette("Paired")

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
sns.set_context('talk',font_scale=2.5)

avg_switch=rt_df.query("stay_num==1").groupby(by=['sub_num','switch']).mean(numeric_only=True).reset_index()

sns.boxplot(x='switch',y='zlog_rt',color='grey',order=['no_switch','switch'],showmeans=True,
            meanprops={"marker":"P",
                       "markerfacecolor":"white", 
                       "markeredgecolor":"black",
                      "markersize":"15"},data=avg_switch)
sns.swarmplot(x='switch',y='zlog_rt',color='white',edgecolor="black",linewidth=2,alpha=0.2,size=10,data=avg_switch)

plt.xticks([0,1],['no switch','switch'])
plt.xlabel('')
plt.ylabel('relative response time')
plt.savefig("plots/switch_zlog_rt.png",bbox_inches = "tight",dpi=300)

In [ ]:
avg_switch=all_data.query("galaxy!=2").groupby(by=['sub_num','switch']).mean().reset_index()

y=avg_switch.query("switch=='no_switch'").prt_rel_om.reset_index(drop=True)
z=avg_switch.query("switch=='switch'").prt_rel_om.reset_index(drop=True)
diff=z-y
stats.ttest_1samp(diff,popmean=0)

In [ ]:
avg_switch=rt_df.query("stay_num==1").groupby(by=['sub_num','switch']).mean().reset_index()

x=avg_switch.query("switch=='switch'").zlog_rt.reset_index(drop=True)-avg_switch.query("switch=='no_switch'").zlog_rt.reset_index(drop=True)
print(stats.ttest_1samp(x,popmean=0,nan_policy='omit'))


temp = pd.DataFrame({'sub_num':avg_switch.query("switch=='switch'").sub_num.reset_index(drop=True),'zlog_rt':x})

In [ ]:
len(np.unique(avg_switch.sub_num))

In [ ]:
x=avg_switch.query("switch=='switch'").zlog_rt.reset_index(drop=True)
y=avg_switch.query("switch=='no_switch'").zlog_rt.reset_index(drop=True)
stats.ttest_rel(x,y)

In [ ]:
print(stats.ttest_1samp(x-y,popmean=0))

In [ ]:
#temp=pd.merge(temp,quic)
params = pd.read_csv("../../../model/fitting/experiment1k_rescorla_wagner/params_val_adaptive_discount_10_10_3.csv")

params=params.join(temp.set_index('sub_num'), on='sub_num').dropna()
#params

In [ ]:
#params = pd.read_csv("../../../model/fitting/experiment1k_rescorla_wagner/params_val_adaptive_discount_10_10_3.csv")
#params=params.join(temp.set_index('sub_num'), on='sub_num').dropna()
params['alphaz']=-np.log(-1 + (10/np.array(params['alpha'])))
params['gamma_basez']=-np.log(-1 + ((12)/(np.array(params['gamma_base']+2))));
params['gamma_coefz']=-np.log(-1 + (6/(np.array(params['gamma_coef']+3))));
x=all_data.groupby(by=['sub_num','switch']).mean().prt_rel_om.reset_index()

y=x.query("switch=='no_switch'").prt_rel_om.reset_index(drop=True)
z=x.query("switch=='switch'").prt_rel_om.reset_index(drop=True)
diff=z-y
params['switch_explicit']= diff


In [ ]:
pg.corr(params['alphaz'],params['zlog_rt'],method='kendall')

In [ ]:
pg.corr(params['gamma_coefz'],params['switch_explicit'],method='kendall')

In [ ]:
sns.set_context('talk',font_scale=1.1)

sns.lmplot(x='alpha',y='zlog_rt',scatter_kws={'color':'k'},line_kws={'color':'k'},data=params)
plt.xlabel("environment complexity prior")
plt.ylabel("switch - no switch RT")
plt.savefig("plots/alpha_zlog_rt_corr.png",bbox_inches = "tight",dpi=300)

In [ ]:
pg.corr(params['alphaz'],params['gamma_coefz'],method='kendall')

In [ ]:
%%R params

params$alpha2 <- params$alphaz^2


#fit quadratic regression model
quadraticModel <- lm(gamma_coefz ~ alpha + alpha2, data=params)

#view model summary
summary(quadraticModel)

In [ ]:
sns.lmplot(x='gamma_coef',y='gamma_base',order=1,scatter_kws={'color':'k'},line_kws={'color':'k'},data=params)

In [ ]:
sns.set_context('talk',font_scale=1.1)

sns.lmplot(x='gamma_coef',y='switch_explicit',scatter_kws={'color':'k'},line_kws={'color':'k'},data=params)
plt.xlabel("uncertainty adaptation parameter")
plt.ylabel("switch - no switch overharvest")
plt.xlim([-1,1])
#plt.ylim([-0.5,0.5])
plt.savefig("plots/gamma_coef_overharvesting_corr.png",bbox_inches = "tight",dpi=300)

In [ ]:
sns.jointplot(x='gamma_basez',y='switch_explicit',data=params)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R -i params


model<-lm(switch_explicit~gamma_coefz,data=params)
summary(model)
plot(model)

In [ ]:
sns.set_context('talk',font_scale=1.0)

sns.lmplot(x='total_score',y='zlog_rt',scatter_kws={'color':'k'},line_kws={'color':'k'},data=params)
plt.xlabel("QUIC")
plt.ylabel("switch - no switch zlogRT")
plt.savefig("plots/quic_zlog_rt_corr.png",bbox_inches = "tight",dpi=300)

In [ ]:

sns.lmplot(x='total_score',y='gamma_coefz',scatter_kws={'color':'k'},line_kws={'color':'k'},data=params)
plt.xlabel("QUIC")
plt.ylabel("gamma uncertainty coef")
plt.savefig("plots/quic_gamma_coef_corr.png",bbox_inches = "tight",dpi=300)

In [ ]:
sns.lmplot(x='total_score',y='gamma_basez',scatter_kws={'color':'k'},line_kws={'color':'k'},data=params)
plt.xlabel("QUIC")
plt.ylabel("gamma base coef")
plt.savefig("plots/quic_gamma_base_corr.png",bbox_inches = "tight",dpi=300)

In [ ]:
def label_quic_group(row):
    if row['total_score'] <8:
        return 'low'
    elif row['total_score'] >8:
        return 'high'
    else:
        return 'median'
    

In [ ]:
params['quic_group'] = params.apply (lambda row: label_quic_group(row), axis=1)
sns.palplot(sns.color_palette("Paired"))

In [ ]:
sns.set_context('talk',font_scale=1.5)
my_pal = sns.color_palette("Paired")
sns.boxplot(x='quic_group',y='zlog_rt',palette=[my_pal[1],my_pal[5],my_pal[9]],data=params)
sns.swarmplot(x='quic_group',y='zlog_rt',size=10,linewidth=2,palette=[my_pal[1],my_pal[5],my_pal[9]],data=params)
plt.ylabel("switch - no switch zlogRT")
plt.xlabel("QUIC group")
plt.savefig("plots/quic_zlog_rt.png",bbox_inches = "tight",dpi=300)

In [ ]:
my_pal = sns.color_palette("Paired")

sns.pointplot(x='quic_group',y='alphaz',palette=[my_pal[1],my_pal[5],my_pal[9]],ci=68,data=params)

In [ ]:
high=params.query("quic_group=='high'").zlog_rt
low=params.query("quic_group=='low'").zlog_rt

stats.ttest_ind(high,low)

In [ ]:
sns.set_context('talk',font_scale=1.2)

sns.lmplot(x='alpha',y='zlog_rt',scatter_kws={'color':'k'},line_kws={'color':'k'},data=params)
plt.ylabel('switch RT - no switch RT')
plt.xlabel('structure learning parameter')
#plt.savefig("plots/alpha_trans_switch_zlog_rt.png",bbox_inches = "tight",dpi=300)

In [ ]:
sns.barplot(x='switch_group',y='zlog_rt',color='white',edgecolor='black',ci=68,data=params)
sns.swarmplot(x='switch_group',y='zlog_rt',color='grey',edgecolor="black",alpha=0.5,data=params)

In [ ]:
pg.corr(params['alphaz'],params['zlog_rt'],method='kendall')

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
sns.set_context('talk',font_scale=2.5)
sns.barplot(y='zlog_rt',color='white',edgecolor='black',ci=68,data=temp)
sns.swarmplot(y='zlog_rt',color='grey',edgecolor="black",alpha=0.5,data=temp)

plt.xlabel('switch - no switch')
plt.ylabel('zlog rt')
plt.savefig("plots/switch_zlog_rt_one_bar.png",bbox_inches = "tight",dpi=300)

In [ ]:
avg = rt_df.groupby(by=['sub_num',"stay?"]).mean().reset_index()


sns.pointplot(x='stay?',y='rt',color='k',ci=68,data=avg)
plt.tight_layout()
plt.savefig("plots/stay_leave_rt.png")

In [ ]:
avg = rt_df.groupby(by=['sub_num',"galaxy"]).mean().reset_index()

sns.pointplot(x='galaxy',y='zlog_rt',ci=68,color='k',data=avg)
plt.tight_layout()

plt.savefig("plots/galaxy_rt.png")

In [ ]:
avg = rt_df.groupby(by=['sub_num',"stay?","galaxy"]).mean().reset_index()

g=sns.pointplot(x='galaxy',y='zlog_rt',hue="stay?",palette=['black','grey'],legend_out=False,ci=68,data=avg)
g.set_xticklabels(['poor','neutral','rich'])

plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
#plt.legend(labels=['leave','stay'])

plt.savefig("plots/galaxy_rt_staay.png")

In [ ]:
avg = rt_df.groupby(by=['sub_num',"num_in_gal"]).mean().reset_index()
sns.pointplot(x='num_in_gal',y='zlog_rt',ci=68,data=avg.query("num_in_gal <= 6"))

In [ ]:
plt.rcParams["figure.figsize"] = (20,10)

avg = rt_df.groupby(by=['sub_num',"num_in_gal","stay?"]).mean().reset_index()
sns.pointplot(x='num_in_gal',y='zlog_rt',hue="stay?",palette=['black','grey'],ci=68,data=avg.query("num_in_gal <= 6"))
plt.xticks(rotation=90)
plt.xlim([-0.5,6.05])

# correlate w/ model parameters

In [ ]:
params = pd.read_csv("params_2022.csv")

rt_df=pd.merge(rt_df, params, on='sub_num')
all_data=pd.merge(all_data, params, on='sub_num')
diff=pd.merge(diff, params, on='sub_num')

In [ ]:
sns.set_context('talk',font_scale=3)

In [ ]:
sns.lmplot(x='gamma',y='total_score',data=all_data)

In [ ]:
sns.set_context('talk',font_scale=2)

avg = all_data.groupby(by=['sub_num','alpha']).mean().reset_index()

sns.lmplot(x='alpha',y='total_reward',data=avg)

In [ ]:
avg = all_data.groupby(by=['sub_num','gamma']).mean().reset_index()

sns.lmplot(x='gamma',y='total_reward',data=avg)

In [ ]:
avg = all_data.groupby(by=['sub_num','alpha']).mean().reset_index()

avg['group'] =avg.apply (lambda row: label_group(row), axis=1)
g=sns.barplot(x='group',y='total_reward',alpha=0.5,ci=68,data=avg)
sns.stripplot(x='group',y='total_reward',s=10,data=avg)
g.set_ylim([12000,15000])
print(stats.ttest_ind(avg.query("group=='high'").total_reward,avg.query("group=='low'").total_reward))
plt.savefig('plots/total_reward_v_alpha_large.png',bbox_inches = "tight",dpi=300)

In [ ]:
avg['rho_2_trans_0'] = rho_2_trans_0
avg['rho_2_trans_1'] = rho_2_trans_1

In [ ]:
g=sns.barplot(x='group',y='rho_2_trans_0',alpha=0.5,ci=68,data=avg)
sns.stripplot(x='group',y='rho_2_trans_0',s=10,data=avg)
g.set_ylabel('correlation coef')
g.set_title('prt changes with probability of transition - rich --> poor')
print(stats.ttest_ind(avg.query("group=='high'").rho_2_trans_0,avg.query("group=='low'").rho_2_trans_0))
plt.savefig('plots/rho_2_0_v_alpha_large.png',bbox_inches = "tight",dpi=300)

In [ ]:
g=sns.barplot(x='group',y='rho_2_trans_1',alpha=0.5,ci=68,data=avg)
sns.stripplot(x='group',y='rho_2_trans_1',s=10,data=avg)
g.set_ylabel('correlation coef')
g.set_title('prt changes with probability of transition - rich --> neutral')
print(stats.ttest_ind(avg.query("group=='high'").rho_2_trans_1,avg.query("group=='low'").rho_2_trans_1))
plt.savefig('plots/rho_2_1_v_alpha.png',bbox_inches = "tight",dpi=300)

In [ ]:
avg = diff.groupby(by=['sub_num','alpha']).mean().reset_index()

avg['group'] =avg.apply (lambda row: label_group(row), axis=1)
g=sns.barplot(x='group',y='diff',alpha=0.5,ci=68,data=avg)
sns.stripplot(x='group',y='diff',s=10,data=avg)
#print(stats.ttest_ind(avg.query("group=='high'").diff,avg.query("group=='low'").diff))
plt.savefig('plots/diff_v_alpha_large.png',bbox_inches = "tight",dpi=300)

In [ ]:
avg=rt_df.groupby(by=['sub_num',"stay?","alpha","num_in_gal"]).mean().reset_index()
avg['stay']=avg['stay?']
leave_rt = avg.query("stay==0 & num_in_gal==6")

sns.lmplot(x='alpha',y='zlog_rt',data=leave_rt)

In [ ]:
def label_group (row):
    if row['alpha'] < 1:
        return 'low'
    else:
        return 'high'
leave_rt['group'] =leave_rt.apply (lambda row: label_group(row), axis=1)

In [ ]:
sns.barplot(x='group',y='zlog_rt',ci=68,alpha=0.5,data=leave_rt)
sns.stripplot(x='group',y='zlog_rt',s=10,data=leave_rt)

print(stats.ttest_ind(leave_rt.query("group=='high'").zlog_rt,leave_rt.query("group=='low'").zlog_rt))
plt.savefig('plots/zlog_rt_v_alpha.png',bbox_inches = "tight",dpi=300)

# Demograpics 

In [ ]:
sns.distplot(all_data.groupby("sub_num").age.mean().tolist())
if savefig:
    plt.savefig("plots/age_dsistribution.png")


print(pg.corr(all_data.groupby("sub_num").age.mean(),all_data.groupby("sub_num").prt_rel_om.mean()))
sns.jointplot(all_data.groupby("sub_num").age.mean(),all_data.groupby("sub_num").prt_rel_om.mean(),kind="reg")
if savefig:
    plt.savefig("plots/age_dsistribution_corr_prt_rel_mvt.png")

In [ ]:
np.std(all_data.groupby("sub_num").age.mean())

In [ ]:
stats.tstd(all_data.groupby(by='sub_num').age.mean())

In [ ]:
sns.jointplot(all_data.groupby("sub_num").age.mean(),all_data.groupby("sub_num").prt_rel_mvt.mean(),kind="reg")

In [ ]:
x=all_data.groupby("sub_num")['gender'].agg(pd.Series.mode).to_frame().reset_index()
x['gender'].value_counts()

In [ ]:
len(np.unique(all_data.sub_num))

In [ ]:
plt.hist(gender_df['gender'].tolist())
if savefig:
    plt.savefig(round+"/hist_gender.png")

# QUIC

In [ ]:
##### FOR ANALYZING QUIC DATA 
def indiv_quic_data(filename,sub_num):
    raw_data = pd.read_csv(filename)
    sub_quic_data = pd.DataFrame(columns=["sub_num",'q_num','answer'])
    prior_12=raw_data.loc[raw_data["trial_type"]=="quic_prior_12"]

    for question in range(6,15):
        temp = re.split(':|{|}',str(prior_12.iloc[:,question].values[0]))
        temp = list(filter(lambda a: a != '', temp))
        q_num = int(temp[0][1])
        temp_df = pd.DataFrame({'sub_num': sub_num,'q_num':q_num,'answer':temp[1]}, index=[0])
        sub_quic_data = sub_quic_data.append(temp_df)
    
    prior_18=raw_data.loc[raw_data["trial_type"]=="quic_prior_18"]
    for question in range(6,35):
        temp = re.split(':|{|}',prior_18.iloc[:,question].values[0])
        temp = list(filter(lambda a: a != '', temp))
        temp[1] = temp[1].split('}')[0]
        q_num = int(temp[0][1:]) + 9

        temp_df = pd.DataFrame({'sub_num': sub_num,'q_num':q_num,'answer':temp[1]}, index=[0])
        sub_quic_data = sub_quic_data.append(temp_df)
    sub_quic_data = sub_quic_data.replace({'Yes':1,'No':0,'Prefer not to answer':np.nan})
    return sub_quic_data

def make_all_quic_data(filenames):
    num_subs = len(filenames)
    quic_data = pd.DataFrame(columns=["sub_num",'q_num','answer','total_score'])
    for f in range(num_subs):
        try:
            sub_data = indiv_quic_data(filenames[f],f)
            summary_score = quic_summary(sub_data)
            sub_data['total_score'] = summary_score
            quic_data = quic_data.append(sub_data)
        except:
            continue
    quic_data = quic_data.astype({'sub_num':'int','q_num':'int','total_score': 'int'})

    return quic_data

def quic_summary(dataframe):
    reverse_items = [0,1,2,3,4,5,6,8,9,10,13,15,25,32]
    # reverse code the dataframes 
    dataframe = reverse_code_items(dataframe,reverse_items)
    # now calc a summary score 
    return dataframe['answer'].sum()
    

def reverse_code_items(dataframe,reverse_items):
    for item in reverse_items:
        dataframe.iloc[item]['answer'] = 1 - dataframe.iloc[item]['answer']
    return dataframe

In [ ]:
all_quic_data = make_all_quic_data(all_filenames)
quic_sub_sort = all_quic_data.sort_values(by=['total_score'],ascending=False)['sub_num'].unique()
# high to low 
quic_sub_sort

#all_quic_data = all_quic_data.dropna()

In [ ]:
quic=all_quic_data.loc[:,['sub_num','total_score']]
quic=quic.groupby(by='sub_num').mean().reset_index()

In [ ]:
x=avgd_data.merge(all_quic_data.loc[:,['sub_num','total_score']].drop_duplicates())
x.loc[:,['sub_num','condition','prt_rel_mvt','total_score']].to_csv('to_correlate_with_params.csv')

In [ ]:
quic.groupby(by='sub_num').mean().reset_index().to_csv("to_correlate.csv")

In [ ]:
avg = all_data.groupby('sub_num').mean().reset_index()
subs = avg["sub_num"].tolist()
avg = avg.set_index('sub_num')

avg = all_data.groupby(['sub_num','galaxy']).mean().reset_index()
avg_rich=avg.query("galaxy==2")
avg_rich = avg_rich.set_index('sub_num')

avg_quic=all_quic_data.groupby('sub_num').mean().reset_index(0)
avg_quic = avg_quic.set_index('sub_num')
quic_df = pd.DataFrame({'sub_num':subs,'avg_prt':list(avg['prt'].loc[subs]),'avg_rich_prt':list(avg_rich['prt'].loc[subs]),'total_reward':list(avg['total_reward'].loc[subs]),'quic_score':list(avg_quic['total_score'].loc[subs])})
quic_df = quic_df[~quic_df['sub_num'].isin(subs_to_drop)]

In [ ]:
sns.distplot(avg_quic['total_score'].tolist())

In [ ]:
x=quic_df.query("quic_score>5")
import pingouin as pg
print(pg.corr(x['quic_score'],x['avg_prt']))

sns.lmplot(x='avg_prt',y='quic_score',data=quic_df.query("quic_score>5"))

# Difference in PRTs in rich galaxies 

In [ ]:
print(pg.corr(quic_df['diff_rich'],quic_df['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich',y='log_quic',kind="reg",data=quic_df)
if savefig:
    plt.savefig(round +"/corr_diff_rich.png")

In [ ]:
savefig=0
print(pg.corr(quic_1['diff_rich'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich',y='log_quic',kind="reg",data=quic_1)
if savefig:
    plt.savefig("plots/corr_diff_rich_cond_1.png")

print(pg.corr(quic_2['diff_rich'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig("plots/corr_diff_rich_cond_2.png")

print(pg.corr(quic_5['diff_rich'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig("plots/corr_diff_rich_cond_5.png")
    
print(pg.corr(quic_6['diff_rich'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig("plots/corr_diff_rich_cond_6.png")

# Difference in poor galaxies 

In [ ]:
print(pg.corr(quic_df['diff_poor'],quic_df['log_quic'],method='spearman'))

sns.jointplot(x='diff_poor',y='log_quic',kind="reg",data=quic_df)
if savefig:
    plt.savefig(round +"/corr_diff_poor.png")

In [ ]:
print(pg.corr(quic_1['diff_poor'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='diff_poor',y='log_quic',kind="reg",data=quic_1)
if savefig:
    plt.savefig("plots/corr_diff_poor_cond_1.png")

print(pg.corr(quic_2['diff_poor'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='diff_poor',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig("plots/corr_diff_poor_cond_2.png")


print(pg.corr(quic_5['diff_poor'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='diff_poor',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig("plots/corr_diff_poor_cond_5.png")
    
print(pg.corr(quic_6['diff_poor'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='diff_poor',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig("plots/corr_diff_poor_cond_6.png")

# Difference in neutral galaxies

In [ ]:
print(pg.corr(quic_df['diff_neut'],quic_df['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut',y='log_quic',kind="reg",data=quic_df)

if savefig:
    plt.savefig(round +"/corr_diff_neut.png")

In [ ]:
print(pg.corr(quic_1['diff_neut'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut',y='quic_score',kind="reg",data=quic_1)
if savefig:
    plt.savefig("plots/corr_diff_neut_cond_1.png")

print(pg.corr(quic_2['diff_neut'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig("plots/corr_diff_neut_cond_2.png")


print(pg.corr(quic_5['diff_neut'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig("plots/corr_diff_neut_cond_5.png")

print(pg.corr(quic_6['diff_neut'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig("plots/corr_diff_neut_cond_6.png")

In [ ]:
print(pg.corr(quic_df['avg_prt'],quic_df['log_quic'],method='spearman'))
sns.jointplot(x='avg_prt',y='log_quic',kind="reg",data=quic_df)
plt.ylabel("QUIC score")
plt.xlabel('planet residence time')
if savefig:
    plt.savefig(round +"/corr_prt_quic_all.png")

In [ ]:
print(pg.corr(quic_1['avg_prt'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='avg_prt',y='log_quic',kind="reg",data=quic_1)
if savefig:
    plt.savefig(round +"/corr_avg_prt_cond_1.png")

print(pg.corr(quic_2['avg_prt'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='avg_prt',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig(round +"/corr_avg_prt_cond_2.png")


print(pg.corr(quic_5['avg_prt'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='avg_prt',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig(round +"/corr_avg-prt_cond_5.png")
    
    
print(pg.corr(quic_6['avg_prt'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='avg_prt',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig(round +"/corr_avg-prt_cond_6.png")

In [ ]:
print(pg.corr(quic_df['neutral_prt'],quic_df['log_quic'],method='spearman'))
sns.jointplot(x='neutral_prt',y='log_quic',kind="reg",data=quic_df)
plt.ylabel("QUIC score")
plt.xlabel('planet residence time')
if savefig:
    plt.savefig(round +"/corr_neutral_prt_quic_all.png")

In [ ]:
print(pg.corr(quic_1['neutral_prt'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='neutral_prt',y='quic_score',kind="reg",data=quic_1)
if savefig:
    plt.savefig("plots/corr_neutral_prt_cond_1.png")

print(pg.corr(quic_2['neutral_prt'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='neutral_prt',y='quic_score',kind="reg",data=quic_2)
if savefig:
    plt.savefig("plots/corr_neutral_prt_cond_2.png")


print(pg.corr(quic_5['neutral_prt'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='neutral_prt',y='quic_score',kind="reg",data=quic_5)
if savefig:
    plt.savefig("plots/corr_neutral_prt_cond_5.png")

print(pg.corr(quic_6['neutral_prt'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='neutral_prt',y='quic_score',kind="reg",data=quic_6)
if savefig:
    plt.savefig("plots/corr_neutral_prt_cond_6.png")

In [ ]:
print(pg.corr(quic_df['poor_prt'],quic_df['log_quic'],method='spearman'))
sns.jointplot(x='poor_prt',y='log_quic',kind="reg",data=quic_df)
plt.ylabel("QUIC score")
plt.xlabel('planet residence time')
if savefig:
    plt.savefig("plots/corr_poor_prt_quic_all.png")

In [ ]:
print(pg.corr(quic_1['poor_prt'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='poor_prt',y='log_quic',kind="reg",data=quic_1)
if savefig:
    plt.savefig(round +"/corr_poor_prt_cond_1.png")

print(pg.corr(quic_2['poor_prt'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='poor_prt',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig(round +"/corr_poor_prt_cond_2.png")


print(pg.corr(quic_5['poor_prt'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='poor_prt',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig(round +"/corr_poor_prt_cond_5.png")
    
    
print(pg.corr(quic_6['poor_prt'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='poor_prt',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig(round +"/corr_poor_prt_cond_6.png")

# Correlations relative to MVT

In [ ]:
x1 = avgd_data_post.query('galaxy == 2 and preced_gal == 0').groupby('sub_num').prt_rel_mvt.mean()
x2 = avgd_data_post.query('galaxy == 2 and preced_gal == 1').groupby('sub_num').prt_rel_mvt.mean()
diff_rich = x1 - x2


x1 = avgd_data_post.query('galaxy == 0 and preced_gal == 1').groupby('sub_num').prt_rel_mvt.mean()
x2 = avgd_data_post.query('galaxy == 0 and preced_gal == 2').groupby('sub_num').prt_rel_mvt.mean()
diff_poor = x1-x2

x1 = avgd_data_post.query('galaxy == 1 and preced_gal == 2').groupby('sub_num').prt_rel_mvt.mean()
x2 = avgd_data_post.query('galaxy == 1 and preced_gal == 0').groupby('sub_num').prt_rel_mvt.mean()
diff_neut = x1-x2


quic_df['diff_rich_rel_mvt'] = diff_rich
quic_df['diff_poor_rel_mvt'] = diff_poor
quic_df['diff_neut_rel_mvt'] = diff_neut
quic_df["neutral_prt_rel_mvt"] = avgd_data_post.query("galaxy == 1").groupby("sub_num").prt_rel_mvt.mean()
quic_df["poor_prt_rel_mvt"] = avgd_data_post.query("galaxy == 0").groupby("sub_num").prt_rel_mvt.mean()

In [ ]:
quic_1 = quic_df.loc[quic_df['condition']==1]
quic_2 = quic_df.loc[quic_df['condition']==2]
quic_5 = quic_df.loc[quic_df['condition']==5]
quic_6 = quic_df.loc[quic_df['condition']==6]

# diff rich

In [ ]:
print(pg.corr(quic_df['diff_rich_rel_mvt'],quic_df['quic_score'],method='spearman'))
sns.jointplot(x='diff_rich_rel_mvt',y='quic_score',kind="reg",data=quic_df)
if savefig:
    plt.savefig(round +"/corr_diff_rich_rel_prt.png")

In [ ]:
print(pg.corr(quic_1['diff_rich_rel_mvt'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_rel_mvt',y='log_quic',kind="reg",data=quic_1)
if savefig:
    plt.savefig("plots/corr_diff_rich_rel_mvt_cond_1.png")

print(pg.corr(quic_2['diff_rich_rel_mvt'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_rel_mvt',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig("plots/corr_diff_rich_rel_mvt_cond_2.png")

print(pg.corr(quic_5['diff_rich_rel_mvt'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_rel_mvt',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig("plots/corr_diff_rich_rel_mvt_cond_5.png")
    
print(pg.corr(quic_6['diff_rich_rel_mvt'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_rel_mvt',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig("plots/corr_diff_rich_rel_mvt_cond_6.png")

# diff poor

In [ ]:
print(pg.corr(quic_df['diff_poor_rel_mvt'],quic_df['log_quic'],method='spearman'))
sns.jointplot(x='diff_poor_rel_mvt',y='log_quic',kind="reg",data=quic_df)
if savefig:
    plt.savefig(round +"/corr_poor_poor_rel_mvt.png")

In [ ]:
print(pg.corr(quic_1['diff_poor_rel_mvt'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='diff_poor_rel_mvt',y='log_quic',kind="reg",data=quic_1)
if savefig:
    plt.savefig("plots/corr_diff_poor_rel_mvt_cond_1.png")

print(pg.corr(quic_2['diff_poor_rel_mvt'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='diff_poor_rel_mvt',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig("plots/corr_diff_poor_rel_mvt_cond_2.png")

print(pg.corr(quic_5['diff_poor_rel_mvt'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='diff_poor_rel_mvt',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig("plots/corr_diff_poor_rel_mvt_cond_5.png")
    
    
print(pg.corr(quic_6['diff_poor_rel_mvt'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='diff_poor_rel_mvt',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig("plots/corr_diff_poor_rel_mvt_cond_6.png")

# diff neutral

In [ ]:
print(pg.corr(quic_df['diff_neut_rel_mvt'],quic_df['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_rel_mvt',y='log_quic',kind="reg",data=quic_df)
if savefig:
    plt.savefig(round +"/corr_diff_neut_rel_mvt.png")

In [ ]:
print(pg.corr(quic_1['diff_neut_rel_mvt'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_rel_mvt',y='log_quic',kind="reg",data=quic_1)
if savefig:
    plt.savefig("plots/corr_diff_neut_rel_mvt_cond_1.png")

print(pg.corr(quic_2['diff_neut_rel_mvt'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_rel_mvt',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig("plots/corr_diff_neut_rel_mvt_cond_2.png")

print(pg.corr(quic_5['diff_neut_rel_mvt'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_rel_mvt',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig("plots/corr_diff_neut_rel_mvt_cond_5.png")
    
print(pg.corr(quic_6['diff_neut_rel_mvt'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_rel_mvt',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig("plots/corr_diff_neut_rel_mvt_cond_6.png")

# difference between average neutral and average poor

In [ ]:
quic_df["diff_neut_poor"] = avgd_data_post.query("galaxy == 1").groupby("sub_num").prt.mean() - avgd_data_post.query("galaxy == 0").groupby("sub_num").prt.mean()

In [ ]:
quic_1 = quic_df.loc[quic_df['condition']==1]
quic_2 = quic_df.loc[quic_df['condition']==2]
quic_5 = quic_df.loc[quic_df['condition']==5]
quic_6 = quic_df.loc[quic_df['condition']==6]

In [ ]:
print(pg.corr(quic_df['diff_neut_poor'],quic_df['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_poor',y='log_quic',kind="reg",data=quic_df)
if savefig:
    plt.savefig(round +"/corr_ddiff_neut_poor.png")

In [ ]:
print(pg.corr(quic_1['diff_neut_poor'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_poor',y='log_quic',kind="reg",data=quic_1)
if savefig:
    plt.savefig("plots/corr_diff_neut_poor_cond_1.png")

print(pg.corr(quic_2['diff_neut_poor'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_poor',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig("plots/corr_diff_neut_poor_cond_2.png")

print(pg.corr(quic_5['diff_neut_poor'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_poor',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig("plots/corr_diff_neut_poor_cond_5.png")
    
    
print(pg.corr(quic_6['diff_neut_poor'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_poor',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig("plots/corr_diff_neut_poor_cond_6.png")

# difference between average neutral and average rich

In [ ]:
quic_df["diff_rich_neut"] = avgd_data_post.query("galaxy == 2").groupby("sub_num").prt.mean() - avgd_data_post.query("galaxy == 1").groupby("sub_num").prt.mean()

In [ ]:
quic_1 = quic_df.loc[quic_df['condition']==1]
quic_2 = quic_df.loc[quic_df['condition']==2]
quic_5 = quic_df.loc[quic_df['condition']==5]
quic_6 = quic_df.loc[quic_df['condition']==6]

In [ ]:
print(pg.corr(quic_df['diff_rich_neut'],quic_df['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_neut',y='log_quic',kind="reg",data=quic_df)
if savefig:
    plt.savefig(round +"/corr_diff_rich_neut.png")

In [ ]:
print(pg.corr(quic_1['diff_rich_neut'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_neut',y='log_quic',kind="reg",data=quic_1)
if savefig:
    plt.savefig("plots/corr_diff_rich_neut_cond_1.png")

print(pg.corr(quic_2['diff_rich_neut'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_neut',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig("plots/corr_diff_rich_neut_cond_2.png")

print(pg.corr(quic_5['diff_rich_neut'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_neut',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig("plots/corr_diff_rich_neut_cond_5.png")
    
print(pg.corr(quic_6['diff_rich_neut'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_neut',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig("plots/corr_diff_rich_neut_cond_6.png")

# difference between average neutral and average poor relative to to mvt

In [ ]:
quic_df["diff_neut_poor_rel_mvt"] = avgd_data_post.query("galaxy == 1").groupby("sub_num").prt_rel_mvt.mean() - avgd_data_post.query("galaxy == 0").groupby("sub_num").prt_rel_mvt.mean()

quic_1 = quic_df.loc[quic_df['condition']==1]
quic_2 = quic_df.loc[quic_df['condition']==2]
quic_5 = quic_df.loc[quic_df['condition']==5]
quic_6 = quic_df.loc[quic_df['condition']==6]

In [ ]:
print(pg.corr(quic_df['diff_neut_poor_rel_mvt'],quic_df['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_poor_rel_mvt',y='log_quic',kind="reg",data=quic_df)
if savefig:
    plt.savefig(round +"/corr_ddiff_neut_poor_rel_mvt.png")

In [ ]:
print(pg.corr(quic_1['diff_neut_poor_rel_mvt'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_poor_rel_mvt',y='log_quic',kind="reg",data=quic_1)
if savefig:
    plt.savefig(round +"/corr_diff_neut_poor_rel_mvt_cond_1.png")

print(pg.corr(quic_2['diff_neut_poor_rel_mvt'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_poor_rel_mvt',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig(round +"/corr_diff_neut_poor_rel_mvt_cond_2.png")

print(pg.corr(quic_5['diff_neut_poor_rel_mvt'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_poor_rel_mvt',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig(round +"/corr_diff_neut_poor_rel_mvt_cond_5.png")
    
print(pg.corr(quic_6['diff_neut_poor_rel_mvt'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='diff_neut_poor_rel_mvt',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig(round +"/corr_diff_neut_poor_rel_mvt_cond_6.png")

# difference between average neutral and average rich relative to mvt

In [ ]:
quic_df["diff_rich_neut_rel_mvt"] = avgd_data_post.query("galaxy == 2").groupby("sub_num").prt_rel_mvt.mean() - avgd_data_post.query("galaxy == 1").groupby("sub_num").prt_rel_mvt.mean()

quic_1 = quic_df.loc[quic_df['condition']==1]
quic_2 = quic_df.loc[quic_df['condition']==2]
quic_5 = quic_df.loc[quic_df['condition']==5]
quic_6 = quic_df.loc[quic_df['condition']==6]

In [ ]:
print(pg.corr(quic_df['diff_rich_neut_rel_mvt'],quic_df['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_neut_rel_mvt',y='log_quic',kind="reg",data=quic_df)
if savefig:
    plt.savefig(round +"/corr_diff_rich_neut_rel_mvt.png")

In [ ]:
print(pg.corr(quic_1['diff_rich_neut_rel_mvt'],quic_1['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_neut_rel_mvt',y='log_quic',kind="reg",data=quic_1)
if savefig:
    plt.savefig(round +"/corr_diff_rich_neut_rel_mvt_cond_1.png")

print(pg.corr(quic_2['diff_rich_neut_rel_mvt'],quic_2['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_neut_rel_mvt',y='log_quic',kind="reg",data=quic_2)
if savefig:
    plt.savefig(round +"/corr_diff_rich_neut_rel_mvt_cond_2.png")

print(pg.corr(quic_5['diff_rich_neut_rel_mvt'],quic_5['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_neut_rel_mvt',y='log_quic',kind="reg",data=quic_5)
if savefig:
    plt.savefig(round +"/corr_diff_rich_neut_rel_mvt_cond_5.png")
    
print(pg.corr(quic_6['diff_rich_neut_rel_mvt'],quic_6['log_quic'],method='spearman'))
sns.jointplot(x='diff_rich_neut_rel_mvt',y='log_quic',kind="reg",data=quic_6)
if savefig:
    plt.savefig(round +"/corr_diff_rich_neut_rel_mvt_cond_6.png")

In [ ]:
quic_df["diff_rich_neut_rel_mvt"] = avgd_data.query("galaxy == 1 & prior_galaxy==2").groupby("sub_num").prt_rel_mvt.mean() - avgd_data.query("galaxy == 1 & prior_galaxy==0").groupby("sub_num").prt_rel_mvt.mean()

In [ ]:
high_quic = quic_df.loc[quic_df['quic_score']>13].reset_index(drop=True)
low_quic = quic_df.loc[quic_df['quic_score']<3].reset_index(drop=True)

In [ ]:
high_quic['group'] = 'high_quic'
low_quic['group'] = 'low_quic'
sep_quic = pd.concat([high_quic,low_quic],ignore_index=True)

In [ ]:
all_data.to_csv('all_data_dataset_2022.csv')

In [ ]:
all_data.query("sub_num==1").condition.tolist()[0]

In [ ]:
def _sim(cond):
    condition = dc.condition_type(cond)
    exp_struc, rho_0,decay = dc.load_data(condition)
    
    total_treasure = 0
    total_time = 10
    max_block_time = 360

    harvest_time = 2
    travel_time =15.5
    iti = 1.5

    k=[1]
    prts=[]
    planet_vis = []

    for block in range(5):
        t = (5.5 + harvest_time + iti)
        planet = 0
        r = 0
        next_reward = rho_0[block][planet]
        block_prts = []
        block_planet_vis = []
        while t < max_block_time:
            total_treasure += next_reward

            if r > 0:
                k.append(decay[block][planet][r-1])

            v_stay = next_reward*(sum(k)/len(k))
            v_leave = ((total_treasure/total_time)*(harvest_time+iti))

            if v_stay > v_leave:
                choice = 1
            else:
                choice=0


            if choice == 0:
                block_prts.append(r)
                r=0
                block_planet_vis.append(planet)
                planet += 1
                next_reward = rho_0[block][planet]
                t += travel_time
                total_time += travel_time
            else:
                next_reward = next_reward*decay[block][planet][r]
                r+=1
                t += (harvest_time + iti)
                total_time += (harvest_time + iti)
        print('global_rr: ' + str(total_treasure/total_time))
        prts.append(block_prts)
        planet_vis.append(block_planet_vis)
    return total_treasure,prts,planet_vis

In [ ]:
cond_num = 1
total_treasure,prts,planet_vis=_sim(cond_num )

condition = dc.condition_type(cond_num )

exp_struc, rho_0,decay = dc.load_data(condition)


plt.plot(dc.flatten_list(prts))
plt.plot([len(prts[0]),len(prts[0])],[0,15],'k--')
plt.plot([len(prts[1])+len(prts[0]),len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[2])+len(prts[1])+len(prts[0]),len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0]),len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[4])+len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0]),len(prts[4])+len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')

for block in range(5):
    for planet in range(len(prts[block])):
        planet_num = planet + sum([planet_vis[i][-1] + 1 for i in range(block)])
        galaxy = dc.flatten_list(exp_struc[block])[planet]
        if galaxy == 0:
            plt.axvspan(planet_num, planet_num+1, facecolor='r', alpha=0.1)
        elif galaxy == 1:
            plt.axvspan(planet_num, planet_num+1, facecolor='b', alpha=0.1)
        elif galaxy == 2:
            plt.axvspan(planet_num, planet_num+1, facecolor='g', alpha=0.1)
print(prts)

In [ ]:
cond_num = 2
total_treasure,prts,planet_vis=_sim(cond_num )

condition = dc.condition_type(cond_num )

exp_struc, rho_0,decay = dc.load_data(condition)


plt.plot(dc.flatten_list(prts))
plt.plot([len(prts[0]),len(prts[0])],[0,15],'k--')
plt.plot([len(prts[1])+len(prts[0]),len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[2])+len(prts[1])+len(prts[0]),len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0]),len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[4])+len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0]),len(prts[4])+len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')

for block in range(5):
    for planet in range(len(prts[block])):
        planet_num = planet + sum([planet_vis[i][-1] + 1 for i in range(block)])
        galaxy = dc.flatten_list(exp_struc[block])[planet]
        if galaxy == 0:
            plt.axvspan(planet_num, planet_num+1, facecolor='r', alpha=0.1)
        elif galaxy == 1:
            plt.axvspan(planet_num, planet_num+1, facecolor='b', alpha=0.1)
        elif galaxy == 2:
            plt.axvspan(planet_num, planet_num+1, facecolor='g', alpha=0.1)
print(prts)

In [ ]:
cond_num = 5
total_treasure,prts,planet_vis=_sim(cond_num )

condition = dc.condition_type(cond_num )

exp_struc, rho_0,decay = dc.load_data(condition)


plt.plot(dc.flatten_list(prts))
plt.plot([len(prts[0]),len(prts[0])],[0,15],'k--')
plt.plot([len(prts[1])+len(prts[0]),len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[2])+len(prts[1])+len(prts[0]),len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0]),len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[4])+len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0]),len(prts[4])+len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')

for block in range(5):
    for planet in range(len(prts[block])):
        planet_num = planet + sum([planet_vis[i][-1] + 1 for i in range(block)])
        galaxy = dc.flatten_list(exp_struc[block])[planet]
        if galaxy == 0:
            plt.axvspan(planet_num, planet_num+1, facecolor='r', alpha=0.1)
        elif galaxy == 1:
            plt.axvspan(planet_num, planet_num+1, facecolor='b', alpha=0.1)
        elif galaxy == 2:
            plt.axvspan(planet_num, planet_num+1, facecolor='g', alpha=0.1)
print(prts)

In [ ]:
cond_num = 6
total_treasure,prts,planet_vis=_sim(cond_num )

condition = dc.condition_type(cond_num )

exp_struc, rho_0,decay = dc.load_data(condition)


plt.plot(dc.flatten_list(prts))
plt.plot([len(prts[0]),len(prts[0])],[0,15],'k--')
plt.plot([len(prts[1])+len(prts[0]),len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[2])+len(prts[1])+len(prts[0]),len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0]),len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')
plt.plot([len(prts[4])+len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0]),len(prts[4])+len(prts[3])+len(prts[2])+len(prts[1])+len(prts[0])],[0,15],'k--')

for block in range(5):
    for planet in range(len(prts[block])):
        planet_num = planet + sum([planet_vis[i][-1] + 1 for i in range(block)])
        galaxy = dc.flatten_list(exp_struc[block])[planet]
        if galaxy == 0:
            plt.axvspan(planet_num, planet_num+1, facecolor='r', alpha=0.1)
        elif galaxy == 1:
            plt.axvspan(planet_num, planet_num+1, facecolor='b', alpha=0.1)
        elif galaxy == 2:
            plt.axvspan(planet_num, planet_num+1, facecolor='g', alpha=0.1)
print(prts)

# bayesian

In [ ]:
import pymc3 as pm
group1 = avgd_data_post.query("condition==5").prt_rel_mvt.tolist()
group2 = avgd_data_post.query("condition==6").prt_rel_mvt.tolist()
pooled = np.concatenate((group1,group2))

In [ ]:
μ_m = pooled.mean()
μ_s = pooled.std() * 2

with pm.Model() as model:
    group1_mean = pm.Normal('group1_mean', mu=μ_m, sd=μ_s)
    group2_mean = pm.Normal('group2_mean', mu=μ_m, sd=μ_s)

σ_low = pooled.std()/1000
σ_high = pooled.std()*1000

with model:
    group1_std = pm.Uniform('group1_std', lower=σ_low, upper=σ_high)
    group2_std = pm.Uniform('group2_std', lower=σ_low, upper=σ_high)
    

with model:
    ν = pm.Exponential('ν_minus_one', 1/ 1/29.) + 1
    
with model:
    λ1 = group1_std**-2
    λ2 = group2_std**-2

    group1 = pm.StudentT('drug', nu=ν, mu=group1_mean, lam=λ1, observed=group1)
    group2 = pm.StudentT('placebo', nu=ν, mu=group2_mean, lam=λ2, observed=group2)

In [ ]:
with model:
    diff_of_means = pm.Deterministic('difference of means', group1_mean - group2_mean)
    diff_of_stds = pm.Deterministic('difference of stds', group1_std - group2_std)
    effect_size = pm.Deterministic('effect size',
                                   diff_of_means / np.sqrt((group1_std**2 + group2_std**2) / 2))
    trace = pm.sample(2000)

In [ ]:

pm.plot_posterior(trace, var_names=['group1_mean','group2_mean', 'group1_std', 'group2_std', 'ν_minus_one'],
                  color='#87ceeb');

In [ ]:

pm.plot_posterior(trace, var_names=['difference of means','difference of stds', 'effect size'],
                  ref_val=0,
                  color='#87ceeb');

In [ ]:

pm.forestplot(trace, var_names=['group1_mean',
                                'group2_mean']);
pm.forestplot(trace, var_names=['group1_std',
                               'group2_std',
                               'ν_minus_one']);

In [ ]:
plt.rcParams["figure.figsize"] = (30,10)


plt.axvspan(0,3,facecolor='r', alpha=0.3)
plt.axvspan(3,4,facecolor='g', alpha=0.3)
plt.axvspan(4,7,facecolor='r', alpha=0.3)
plt.axvspan(7,8,facecolor='g', alpha=0.3)
plt.axvspan(8,11,facecolor='r', alpha=0.3)

plt.axvspan(11,12,facecolor='g', alpha=0.3)
plt.axvspan(12,15,facecolor='r', alpha=0.3)
plt.axvspan(15,16,facecolor='g', alpha=0.3)
plt.axvspan(16,19,facecolor='r', alpha=0.3)
plt.axvspan(19,21,facecolor='b', alpha=0.3)

plt.plot([11,11],[0,1],'k--',linewidth=15)
sns.despine(top=True,right=True,bottom=True,left=True)
plt.xticks([])
plt.yticks([])

plt.savefig("plots/exp_struc_poor_extreme_ex.png")

In [ ]:
plt.rcParams["figure.figsize"] = (30,10)


plt.axvspan(0,3,facecolor='g', alpha=0.3)
plt.axvspan(3,4,facecolor='r', alpha=0.3)
plt.axvspan(4,7,facecolor='g', alpha=0.3)
plt.axvspan(7,8,facecolor='r', alpha=0.3)
plt.axvspan(8,11,facecolor='g', alpha=0.3)

plt.axvspan(11,12,facecolor='r', alpha=0.3)
plt.axvspan(12,15,facecolor='g', alpha=0.3)
plt.axvspan(15,16,facecolor='r', alpha=0.3)
plt.axvspan(16,19,facecolor='g', alpha=0.3)
plt.axvspan(19,21,facecolor='b', alpha=0.3)

plt.plot([11,11],[0,1],'k--',linewidth=15)
sns.despine(top=True,right=True,bottom=True,left=True)
plt.xticks([])
plt.yticks([])


plt.savefig("plots/exp_struc_rich_extreme_ex.png")

In [ ]:
plt.rcParams["figure.figsize"] = (30,10)


plt.axvspan(0,5,facecolor='g', alpha=0.6)
plt.axvspan(5,8,facecolor='r', alpha=0.6)
plt.axvspan(8,12,facecolor='g', alpha=0.6)
plt.axvspan(12,16,facecolor='r', alpha=0.6)
plt.axvspan(16,19,facecolor='g', alpha=0.6)
plt.axvspan(19,22,facecolor='b', alpha=0.1)

plt.plot([12,12],[0,1],'k--',linewidth=15)
sns.despine(top=True,right=True,bottom=True,left=True)
plt.xticks([])
plt.yticks([])

plt.savefig("plots/exp_struc_rich_extreme_ex.png")

In [ ]:
plt.rcParams["figure.figsize"] = (30,10)


plt.axvspan(0,5,facecolor='g', alpha=0.3)
plt.axvspan(5,6,facecolor='b', alpha=0.3)
plt.axvspan(6,7,facecolor='r', alpha=0.3)
plt.axvspan(7,8,facecolor='r', alpha=0.3)
plt.axvspan(8,9,facecolor='b', alpha=0.3)
plt.axvspan(9,14,facecolor='g', alpha=0.3)

plt.plot([7,7],[0,1],'k--',linewidth=15)
sns.despine(top=True,right=True,bottom=True,left=True)
plt.xticks([])
plt.yticks([])

plt.savefig("plots/exp_struc_rich_graded_ex.png")

In [ ]:
plt.axvspan(0,5,facecolor='r', alpha=0.3)
plt.axvspan(5,6,facecolor='b', alpha=0.3)
plt.axvspan(6,7,facecolor='g', alpha=0.3)
plt.axvspan(7,8,facecolor='g', alpha=0.3)
plt.axvspan(8,9,facecolor='b', alpha=0.3)
plt.axvspan(9,14,facecolor='r', alpha=0.3)

plt.plot([7,7],[0,1],'k--',linewidth=15)
sns.despine(top=True,right=True,bottom=True,left=True)
plt.xticks([])
plt.yticks([])

plt.savefig("plots/exp_struc_poor_graded_ex.png")

In [ ]:
plt.rcParams["figure.figsize"] = (30,10)


plt.axvspan(0,5,facecolor='r', alpha=0.3)
plt.axvspan(5,8,facecolor='g', alpha=0.3)
plt.axvspan(8,12,facecolor='r', alpha=0.3)

plt.axvspan(12,16,facecolor='g', alpha=0.3)
plt.axvspan(16,19,facecolor='r', alpha=0.3)
plt.axvspan(19,22,facecolor='b', alpha=0.3)

plt.axvspan(22,25,facecolor='b', alpha=0.3)
plt.axvspan(25,29,facecolor='r', alpha=0.3)
plt.axvspan(29,33,facecolor='g', alpha=0.3)

plt.axvspan(33,36,facecolor='g', alpha=0.3)
plt.axvspan(36,40,facecolor='r', alpha=0.3)
plt.axvspan(40,45,facecolor='b', alpha=0.3)

plt.axvspan(45,48,facecolor='b', alpha=0.3)
plt.axvspan(48,52,facecolor='g', alpha=0.3)
plt.axvspan(52,57,facecolor='r', alpha=0.3)

plt.plot([12,12],[0,1],'k--',linewidth=15)
plt.plot([22,22],[0,1],'k--',linewidth=15)
plt.plot([33,33],[0,1],'k--',linewidth=15)
plt.plot([45,45],[0,1],'k--',linewidth=15)



sns.despine(top=True,right=True,bottom=True,left=True)
plt.xticks([])
plt.yticks([])

plt.savefig("plots/whole.png")

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
sns.set_context('talk',font_scale=4)

x = range(100)
y = [1*((0.3)**(i)) for i in x]

plt.plot(x,y,linewidth=15,color=pal[0])
plt.xlim([0,15])
sns.despine(right=True,top=True)
plt.ylabel('Value')
plt.yticks([])
plt.xlabel('Time')
plt.xticks([])

plt.savefig("plots/poor.png")

In [ ]:

x = range(100)
y = [1*((0.6)**(i)) for i in x]

plt.plot(x,y,linewidth=15,color= sns.color_palette("Blues_r", 2)[0])
plt.xlim([0,15])
sns.despine(right=True,top=True)
plt.ylabel('Value')
plt.yticks([])
plt.xlabel('Time')
plt.xticks([])

plt.savefig("plots/neutral.png")

In [ ]:

x = range(100)
y = [1*((0.9)**(i)) for i in x]

plt.plot(x,y,linewidth=15,color= pal[2])
plt.xlim([0,15])
sns.despine(right=True,top=True)
plt.ylabel('Value')
plt.yticks([])
plt.xlabel('Time')
plt.xticks([])

plt.savefig("plots/rich.png")

In [ ]:
avgd_data_post.groupby("condition").prt_rel_mvt.mean().reset_index(drop=False).to_csv("fit_plot_condition_average.csv")

# sim for determining travel time

In [ ]:
import sim 
import pandas as pd
import seaborn as sns

def run_sim(cond_num):
    total_treasure,prts = sim.sim(cond_num)

    block_n_planets = [len(x) for x in prts] 
    exp_struc, rho_0, all_decay = sim.load_data(sim.condition_type(cond_num))
    exp_struc = [sim.flatten_list(x) for x in exp_struc]
    exp_planets = [exp_struc[i][:block_n_planets[i]] for i in range(5)]
    
    post_prts = prts[2:]
    exp_planets = exp_planets[2:]

    exp_planets = sim.flatten_list(exp_planets)
    exp_struc = sim.flatten_list(exp_struc)

    df = pd.DataFrame({'condition':[cond_num]*len(exp_planets),'galaxy':exp_planets,'prt':sim.flatten_list(post_prts)})

    return df

In [ ]:
df_1 = run_sim(1)
sns.barplot(x='galaxy',y='prt',data=df_1)

In [ ]:
df_2 = run_sim(2)
sns.barplot(x='galaxy',y='prt',data=df_2)

In [ ]:
df_5 = run_sim(5)
sns.barplot(x='galaxy',y='prt',data=df_5)

In [ ]:
df_6 = run_sim(6)
sns.barplot(x='galaxy',y='prt',data=df_6)